In [1]:
# ============================================================
# NOTE 08: Chuyển dịch cơ cấu điện có đi kèm thay đổi về phát thải,
# cường độ carbon và năng lực phát điện hay không?
#
# Mục đích: Chuẩn bị cho việc khai thác các biến còn thiếu: Capacity, Emissions, Emissions intensity,
#  YoY changes và các biến geography/group membership.
# ============================================================

import pickle
from pathlib import Path

# ------------------------------------------------------------
# 1. Tìm project_root từ checkpoint tương đối
# ------------------------------------------------------------

relative_note03 = Path("checkpoints/note03_eda_overview.pkl")
relative_note07 = Path("checkpoints/note07_relationship_deeper_analysis.pkl")

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (
            (path / relative_note03).is_file()
            and (path / relative_note07).is_file()
        )
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy project_root chứa checkpoint NOTE 03 và NOTE 07."
    )

note03_path = project_root / relative_note03
note07_path = project_root / relative_note07

# ------------------------------------------------------------
# 2. Load dataframe gốc từ NOTE 03
# ------------------------------------------------------------

with open(note03_path, "rb") as f:
    note03 = pickle.load(f)

df = note03["df"].copy()

# ------------------------------------------------------------
# 3. Load kết quả NOTE 07
# ------------------------------------------------------------

with open(note07_path, "rb") as f:
    note07 = pickle.load(f)

profile = note07["profile"].copy()
transition_features = note07["transition_features"].copy()

# ------------------------------------------------------------
# 4. Kiểm tra các biến gốc
# ------------------------------------------------------------

print("=" * 80)
print("LOAD FROM CHECKPOINTS")
print("=" * 80)

print(f"\nProject root: {project_root}")

print("\n--- NOTE 03 ---")
print(f"Checkpoint: {note03_path}")
print(f"Dataframe shape: {df.shape}")
print(f"Số biến: {len(df.columns)}")

print("\n--- NOTE 07 ---")
print(f"Checkpoint: {note07_path}")
print(f"profile: {profile.shape}")
print(f"transition_features: {transition_features.shape}")

print("\n--- TOÀN BỘ BIẾN GỐC ---")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\n✓ Đã load dataframe gốc từ checkpoint NOTE 03.")
print("✓ Sẵn sàng kiểm tra các biến chưa được khai thác.")

LOAD FROM CHECKPOINTS

Project root: d:\phan tich data\Ember_Electricity

--- NOTE 03 ---
Checkpoint: d:\phan tich data\Ember_Electricity\checkpoints\note03_eda_overview.pkl
Dataframe shape: (104203, 24)
Số biến: 24

--- NOTE 07 ---
Checkpoint: d:\phan tich data\Ember_Electricity\checkpoints\note07_relationship_deeper_analysis.pkl
profile: (209, 13)
transition_features: (209, 7)

--- TOÀN BỘ BIẾN GỐC ---
01. Area
02. ISO 3 code
03. Year
04. Area type
05. Electricity source
06. Is aggregated source
07. Generation (TWh)
08. Generation YoY change (TWh)
09. Generation YoY change (%)
10. Share of generation (%)
11. Share of generation YoY change (% points)
12. Capacity (GW)
13. Emissions (MtCO2e)
14. Emissions YoY change (MtCO2e)
15. Emissions YoY change (%)
16. Share of emissions (%)
17. Emissions intensity (gCO2e/kWh)
18. Continent
19. Ember region
20. EU member
21. OECD member
22. G20 member
23. G7 member
24. ASEAN member

✓ Đã load dataframe gốc từ checkpoint NOTE 03.
✓ Sẵn sàng kiểm tr

In [2]:
# ============================================================
# CHECK COVERAGE OF UNUSED VARIABLES
#
# Các biến chưa khai thác có dữ liệu đầy đủ đến đâu?
# Missingness có mang tính cấu trúc theo source / area type
# hay không?
#
# Mục đích:
# - Không mặc định rằng biến nào cũng dùng được như nhau.
# - Kiểm tra missing rate.
# - Kiểm tra số dòng có dữ liệu.
# - Xác định biến nào phù hợp cho phân tích country-level.
# - Phát hiện biến có missingness do cấu trúc dataset.
# ============================================================

import pandas as pd

unused_cols = [
    "Generation YoY change (TWh)",
    "Generation YoY change (%)",
    "Share of generation YoY change (% points)",
    "Capacity (GW)",
    "Emissions (MtCO2e)",
    "Emissions YoY change (MtCO2e)",
    "Emissions YoY change (%)",
    "Share of emissions (%)",
    "Emissions intensity (gCO2e/kWh)",
    "Continent",
    "Ember region",
    "EU member",
    "OECD member",
    "G20 member",
    "G7 member",
    "ASEAN member",
]

# ------------------------------------------------------------
# 1. Tổng quan coverage
# ------------------------------------------------------------

coverage = pd.DataFrame({
    "Variable": unused_cols,
    "Non_missing": [
        df[col].notna().sum()
        for col in unused_cols
    ],
    "Missing": [
        df[col].isna().sum()
        for col in unused_cols
    ],
})

coverage["Missing_%"] = (
    coverage["Missing"] / len(df) * 100
).round(2)

coverage["Coverage_%"] = (
    coverage["Non_missing"] / len(df) * 100
).round(2)

coverage = coverage.sort_values(
    "Missing_%" if "Missing_%" in coverage.columns else "Missing",
    ascending=False
).reset_index(drop=True)

print("=" * 80)
print("VARIABLE COVERAGE")
print("=" * 80)

print(f"\nTổng số dòng dataset: {len(df):,}")
print(f"Số biến được kiểm tra: {len(unused_cols)}")

print("\n--- COVERAGE TOÀN DATASET ---")
print(coverage.to_string(index=False))

# ------------------------------------------------------------
# 2. Coverage theo Area type
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COVERAGE THEO AREA TYPE")
print("=" * 80)

area_type_coverage = []

for area_type, group in df.groupby("Area type", dropna=False):

    for col in unused_cols:
        area_type_coverage.append({
            "Area_type": area_type,
            "Variable": col,
            "Rows": len(group),
            "Missing_%": round(
                group[col].isna().mean() * 100,
                2
            ),
        })

area_type_coverage = pd.DataFrame(area_type_coverage)

print(
    area_type_coverage
    .pivot(
        index="Variable",
        columns="Area_type",
        values="Missing_%"
    )
    .round(2)
    .to_string()
)

# ------------------------------------------------------------
# 3. Kiểm tra coverage riêng nhóm country/economy
# ------------------------------------------------------------

country_df = df[
    df["Area type"] == "Country or economy"
].copy()

print("\n" + "=" * 80)
print("COVERAGE — COUNTRY / ECONOMY")
print("=" * 80)

country_coverage = pd.DataFrame({
    "Variable": unused_cols,
    "Non_missing": [
        country_df[col].notna().sum()
        for col in unused_cols
    ],
    "Missing": [
        country_df[col].isna().sum()
        for col in unused_cols
    ],
})

country_coverage["Missing_%"] = (
    country_coverage["Missing"] /
    len(country_df) * 100
).round(2)

country_coverage["Coverage_%"] = (
    country_coverage["Non_missing"] /
    len(country_df) * 100
).round(2)

print(country_coverage.to_string(index=False))

print("→ Chưa kết luận biến nào nên/không nên dùng.")
print("→ Bước tiếp theo sẽ dựa trên coverage thực tế ở trên.")

VARIABLE COVERAGE

Tổng số dòng dataset: 104,203
Số biến được kiểm tra: 16

--- COVERAGE TOÀN DATASET ---
                                 Variable  Non_missing  Missing  Missing_%  Coverage_%
                 Emissions YoY change (%)        55442    48761      46.79       53.21
          Emissions intensity (gCO2e/kWh)        63453    40750      39.11       60.89
                Generation YoY change (%)        67000    37203      35.70       64.30
                            Capacity (GW)        75897    28306      27.16       72.84
            Emissions YoY change (MtCO2e)        94378     9825       9.43       90.57
                   Share of emissions (%)        97866     6337       6.08       93.92
                       Emissions (MtCO2e)        97898     6305       6.05       93.95
                                Continent        97937     6266       6.01       93.99
                             Ember region        97937     6266       6.01       93.99
Share of generation YoY 

In [3]:
# ============================================================
# COVERAGE 2012–2022
#
# Trong đúng giai đoạn 2012–2022 của Note 06, các biến mới
# có đủ dữ liệu để so sánh 209 quốc gia hay không?
#
# Mục đích:
# - Giữ cùng một khung thời gian với Note 06–07.
# - Không chọn biến chỉ vì coverage toàn dataset cao.
# - Kiểm tra số quốc gia có dữ liệu ở cả 2012 và 2022.
# - Tìm biến nào có thể tạo transition features mới.
# ============================================================

analysis_years = [2012, 2022]

country_period = df[
    (df["Area type"] == "Country or economy") &
    (df["Year"].isin(analysis_years))
].copy()

analysis_variables = [
    "Generation YoY change (TWh)",
    "Generation YoY change (%)",
    "Share of generation YoY change (% points)",
    "Capacity (GW)",
    "Emissions (MtCO2e)",
    "Emissions YoY change (MtCO2e)",
    "Emissions YoY change (%)",
    "Share of emissions (%)",
    "Emissions intensity (gCO2e/kWh)",
]

print("=" * 80)
print("COVERAGE TRONG GIAI ĐOẠN 2012–2022")
print("=" * 80)

print(f"\nCountry/economy rows: {len(country_period):,}")
print(
    f"Số quốc gia/economy khác nhau: "
    f"{country_period['Area'].nunique()}"
)

# ------------------------------------------------------------
# 1. Coverage từng năm
# ------------------------------------------------------------

year_coverage = []

for year in analysis_years:
    year_df = country_period[country_period["Year"] == year]

    for col in analysis_variables:
        available = year_df.loc[
            year_df[col].notna(), "Area"
        ].nunique()

        year_coverage.append({
            "Year": year,
            "Variable": col,
            "Countries_available": available,
            "Total_countries": year_df["Area"].nunique(),
            "Coverage_%": round(
                available / year_df["Area"].nunique() * 100,
                2
            ),
        })

year_coverage = pd.DataFrame(year_coverage)

print("\n" + "=" * 80)
print("COVERAGE THEO NĂM")
print("=" * 80)

print(
    year_coverage
    .pivot(
        index="Variable",
        columns="Year",
        values="Coverage_%"
    )
    .round(2)
    .to_string()
)

# ------------------------------------------------------------
# 2. Coverage đồng thời 2012 VÀ 2022
# ------------------------------------------------------------

paired_coverage = []

for col in analysis_variables:

    available_by_year = {}

    for year in analysis_years:
        available_by_year[year] = set(
            country_period.loc[
                (country_period["Year"] == year) &
                (country_period[col].notna()),
                "Area"
            ]
        )

    both_years = (
        available_by_year[2012]
        & available_by_year[2022]
    )

    paired_coverage.append({
        "Variable": col,
        "2012_available": len(available_by_year[2012]),
        "2022_available": len(available_by_year[2022]),
        "Both_2012_2022": len(both_years),
        "Paired_coverage_%": round(
            len(both_years) / 209 * 100,
            2
        ),
    })

paired_coverage = pd.DataFrame(paired_coverage)

print("\n" + "=" * 80)
print("COVERAGE ĐỒNG THỜI 2012–2022")
print("=" * 80)

print(
    paired_coverage.to_string(index=False)
)

# ------------------------------------------------------------
# 3. Xác định các biến đủ mạnh để phân tích tiếp
# ------------------------------------------------------------

usable_variables = paired_coverage[
    paired_coverage["Paired_coverage_%"] >= 80
]["Variable"].tolist()

conditional_variables = paired_coverage[
    (paired_coverage["Paired_coverage_%"] >= 50) &
    (paired_coverage["Paired_coverage_%"] < 80)
]["Variable"].tolist()

low_coverage_variables = paired_coverage[
    paired_coverage["Paired_coverage_%"] < 50
]["Variable"].tolist()

print("\n" + "=" * 80)
print("PHÂN NHÓM SAU KHI KIỂM TRA")
print("=" * 80)

print("\n[ƯU TIÊN — >= 80%]")
for col in usable_variables:
    print(f"- {col}")

print("\n[CÓ ĐIỀU KIỆN — 50% đến <80%]")
for col in conditional_variables:
    print(f"- {col}")

print("\n[KHÔNG ƯU TIÊN CHO SO SÁNH 2012–2022]")
for col in low_coverage_variables:
    print(f"- {col}")

print("→ Quyết định biến phân tích sẽ dựa trên paired coverage 2012–2022.")

COVERAGE TRONG GIAI ĐOẠN 2012–2022

Country/economy rows: 6,982
Số quốc gia/economy khác nhau: 209

COVERAGE THEO NĂM
Year                                         2012   2022
Variable                                                
Capacity (GW)                              100.00  100.0
Emissions (MtCO2e)                         100.00  100.0
Emissions YoY change (%)                    99.52  100.0
Emissions YoY change (MtCO2e)               99.52  100.0
Emissions intensity (gCO2e/kWh)            100.00  100.0
Generation YoY change (%)                   99.52  100.0
Generation YoY change (TWh)                 99.52  100.0
Share of emissions (%)                     100.00  100.0
Share of generation YoY change (% points)   99.52  100.0

COVERAGE ĐỒNG THỜI 2012–2022
                                 Variable  2012_available  2022_available  Both_2012_2022  Paired_coverage_%
              Generation YoY change (TWh)             208             209             208              99.52
       

In [4]:
# ============================================================
# TẠO CARBON + CAPACITY TRANSITION FEATURES
#
# Trong 209 quốc gia, từ 2012 đến 2022:
# - Generation thay đổi thế nào?
# - Capacity thay đổi thế nào?
# - Emissions thay đổi thế nào?
# - Emissions intensity thay đổi thế nào?
# - Share of emissions thay đổi thế nào?
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Chọn đúng 209 quốc gia và 2 mốc 2012–2022
# ------------------------------------------------------------
country_2012_2022 = df[
    (df["Area type"] == "Country or economy") &
    (df["Year"].isin([2012, 2022]))
].copy()

print("=" * 80)
print("CARBON + CAPACITY TRANSITION FEATURES")
print("=" * 80)

print(f"\nRows sử dụng: {len(country_2012_2022):,}")
print(f"Countries: {country_2012_2022['Area'].nunique()}")

# ------------------------------------------------------------
# 2. Lấy các chỉ số hệ thống ở nguồn "Total generation"
# ------------------------------------------------------------
system_variables = [
    "Generation (TWh)",
    "Generation YoY change (TWh)",
    "Generation YoY change (%)",
    "Capacity (GW)",
    "Emissions (MtCO2e)",
    "Emissions YoY change (MtCO2e)",
    "Emissions YoY change (%)",
    "Share of emissions (%)",
    "Emissions intensity (gCO2e/kWh)",
]

system_df = country_2012_2022[
    country_2012_2022["Electricity source"] == "Total generation"
].copy()

print(
    f"\nRows Total generation: {len(system_df):,}"
)

print(
    f"Countries trong Total generation: "
    f"{system_df['Area'].nunique()}"
)

# ------------------------------------------------------------
# 3. Pivot 2012 và 2022 thành từng cột
# ------------------------------------------------------------
transition = (
    system_df[
        ["Area", "Year"] + system_variables
    ]
    .pivot(
        index="Area",
        columns="Year",
        values=system_variables
    )
)

# Làm phẳng MultiIndex columns
transition.columns = [
    f"{variable}_{year}"
    for variable, year in transition.columns
]

transition = transition.reset_index()

# ------------------------------------------------------------
# 4. Tạo absolute change và percentage change
# ------------------------------------------------------------

transition["Delta_Generation_TWh"] = (
    transition["Generation (TWh)_2022"]
    - transition["Generation (TWh)_2012"]
)

transition["Delta_Capacity_GW"] = (
    transition["Capacity (GW)_2022"]
    - transition["Capacity (GW)_2012"]
)

transition["Delta_Emissions_MtCO2e"] = (
    transition["Emissions (MtCO2e)_2022"]
    - transition["Emissions (MtCO2e)_2012"]
)

transition["Delta_Emissions_Intensity"] = (
    transition["Emissions intensity (gCO2e/kWh)_2022"]
    - transition["Emissions intensity (gCO2e/kWh)_2012"]
)

transition["Delta_Share_Emissions_pp"] = (
    transition["Share of emissions (%)_2022"]
    - transition["Share of emissions (%)_2012"]
)

# ------------------------------------------------------------
# 5. Tạo % thay đổi cho Generation, Capacity và Emissions
# ------------------------------------------------------------
transition["Generation_Change_%"] = np.where(
    transition["Generation (TWh)_2012"] != 0,
    (
        transition["Delta_Generation_TWh"]
        / transition["Generation (TWh)_2012"]
        * 100
    ),
    np.nan
)

transition["Capacity_Change_%"] = np.where(
    transition["Capacity (GW)_2012"] != 0,
    (
        transition["Delta_Capacity_GW"]
        / transition["Capacity (GW)_2012"]
        * 100
    ),
    np.nan
)

transition["Emissions_Change_%"] = np.where(
    transition["Emissions (MtCO2e)_2012"] != 0,
    (
        transition["Delta_Emissions_MtCO2e"]
        / transition["Emissions (MtCO2e)_2012"]
        * 100
    ),
    np.nan
)

# ------------------------------------------------------------
# 6. Kiểm tra kết quả
# ------------------------------------------------------------
feature_columns = [
    "Area",
    "Delta_Generation_TWh",
    "Generation_Change_%",
    "Delta_Capacity_GW",
    "Capacity_Change_%",
    "Delta_Emissions_MtCO2e",
    "Emissions_Change_%",
    "Delta_Emissions_Intensity",
    "Delta_Share_Emissions_pp",
]

transition_features_carbon = transition[
    feature_columns
].copy()

print("\n" + "=" * 80)
print("KÍCH THƯỚC FEATURE TABLE")
print("=" * 80)

print(
    f"Shape: {transition_features_carbon.shape}"
)

print(
    f"Số quốc gia: "
    f"{transition_features_carbon['Area'].nunique()}"
)

print("\nMissing theo biến:")
print(
    transition_features_carbon
    .isna()
    .sum()
    .to_string()
)

print("\n" + "=" * 80)
print("5 DÒNG ĐẦU")
print("=" * 80)

print(
    transition_features_carbon
    .head()
    .round(3)
    .to_string(index=False)
)

print(
    "→ Đã tạo bộ feature chuyển dịch Generation + Capacity + Emissions + Carbon Intensity."
)
print(
    "→ Tiếp theo kiểm tra các feature này thay đổi như thế nào giữa 209 quốc gia."
)

CARBON + CAPACITY TRANSITION FEATURES

Rows sử dụng: 6,982
Countries: 209

Rows Total generation: 418
Countries trong Total generation: 209

KÍCH THƯỚC FEATURE TABLE
Shape: (209, 9)
Số quốc gia: 209

Missing theo biến:
Area                         0
Delta_Generation_TWh         0
Generation_Change_%          0
Delta_Capacity_GW            0
Capacity_Change_%            3
Delta_Emissions_MtCO2e       0
Emissions_Change_%           0
Delta_Emissions_Intensity    0
Delta_Share_Emissions_pp     0

5 DÒNG ĐẦU
          Area  Delta_Generation_TWh  Generation_Change_%  Delta_Capacity_GW  Capacity_Change_%  Delta_Emissions_MtCO2e  Emissions_Change_%  Delta_Emissions_Intensity  Delta_Share_Emissions_pp
   Afghanistan                  0.08                9.091               0.27             62.791                  -0.013             -10.156                    -25.663                     0.382
       Albania                  2.25               47.669               0.96             55.491         

In [5]:
# ============================================================
# PHÂN PHỐI CARBON + CAPACITY TRANSITION
#
# 1. Phát thải tăng hay giảm ở phần lớn quốc gia?
# 2. Cường độ carbon tăng hay giảm?
# 3. Công suất và sản lượng điện tăng ở mức nào?
# 4. Có quốc gia nào vừa tăng generation nhưng giảm emissions hoặc giảm carbon intensity hay không?
# ============================================================

carbon_features = transition_features_carbon.copy()

# ------------------------------------------------------------
# 1. Hàm thống kê mô tả
# ------------------------------------------------------------
summary_variables = [
    "Delta_Generation_TWh",
    "Generation_Change_%",
    "Delta_Capacity_GW",
    "Capacity_Change_%",
    "Delta_Emissions_MtCO2e",
    "Emissions_Change_%",
    "Delta_Emissions_Intensity",
    "Delta_Share_Emissions_pp",
]

summary = (
    carbon_features[summary_variables]
    .describe()
    .T
)

summary["missing"] = (
    carbon_features[summary_variables]
    .isna()
    .sum()
)

print("=" * 80)
print("PHÂN PHỐI CARBON + CAPACITY TRANSITION")
print("=" * 80)

print("\nTHỐNG KÊ MÔ TẢ:")
print(
    summary[
        ["count", "mean", "std", "min", "25%", "50%", "75%", "max", "missing"]
    ]
    .round(3)
    .to_string()
)

# ------------------------------------------------------------
# 2. Đếm hướng thay đổi
# ------------------------------------------------------------
direction_summary = pd.DataFrame({
    "Metric": [
        "Generation",
        "Capacity",
        "Emissions",
        "Carbon intensity",
        "Share of emissions",
    ],
    "Decrease": [
        (carbon_features["Delta_Generation_TWh"] < 0).sum(),
        (carbon_features["Delta_Capacity_GW"] < 0).sum(),
        (carbon_features["Delta_Emissions_MtCO2e"] < 0).sum(),
        (carbon_features["Delta_Emissions_Intensity"] < 0).sum(),
        (carbon_features["Delta_Share_Emissions_pp"] < 0).sum(),
    ],
    "No_change": [
        (carbon_features["Delta_Generation_TWh"] == 0).sum(),
        (carbon_features["Delta_Capacity_GW"] == 0).sum(),
        (carbon_features["Delta_Emissions_MtCO2e"] == 0).sum(),
        (carbon_features["Delta_Emissions_Intensity"] == 0).sum(),
        (carbon_features["Delta_Share_Emissions_pp"] == 0).sum(),
    ],
    "Increase": [
        (carbon_features["Delta_Generation_TWh"] > 0).sum(),
        (carbon_features["Delta_Capacity_GW"] > 0).sum(),
        (carbon_features["Delta_Emissions_MtCO2e"] > 0).sum(),
        (carbon_features["Delta_Emissions_Intensity"] > 0).sum(),
        (carbon_features["Delta_Share_Emissions_pp"] > 0).sum(),
    ],
})

print("\n" + "=" * 80)
print("HƯỚNG THAY ĐỔI 2012 → 2022")
print("=" * 80)

print(direction_summary.to_string(index=False))

# ------------------------------------------------------------
# 3. Pattern quan trọng:
#    Generation tăng nhưng emissions giảm
# ------------------------------------------------------------
generation_up_emissions_down = carbon_features[
    (carbon_features["Delta_Generation_TWh"] > 0) &
    (carbon_features["Delta_Emissions_MtCO2e"] < 0)
].copy()

generation_up_intensity_down = carbon_features[
    (carbon_features["Delta_Generation_TWh"] > 0) &
    (carbon_features["Delta_Emissions_Intensity"] < 0)
].copy()

generation_up_both_down = carbon_features[
    (carbon_features["Delta_Generation_TWh"] > 0) &
    (carbon_features["Delta_Emissions_MtCO2e"] < 0) &
    (carbon_features["Delta_Emissions_Intensity"] < 0)
].copy()

print("\n" + "=" * 80)
print("PATTERN: GENERATION TĂNG NHƯNG CARBON GIẢM")
print("=" * 80)

print(
    f"\nGeneration tăng + Emissions giảm: "
    f"{len(generation_up_emissions_down)} quốc gia"
)

print(
    f"Generation tăng + Carbon intensity giảm: "
    f"{len(generation_up_intensity_down)} quốc gia"
)

print(
    f"Generation tăng + Emissions giảm + Carbon intensity giảm: "
    f"{len(generation_up_both_down)} quốc gia"
)

# ------------------------------------------------------------
# 4. Hiển thị các trường hợp tiêu biểu
# ------------------------------------------------------------
if len(generation_up_both_down) > 0:
    print("\nTOP 15 QUỐC GIA THEO MỨC GIẢM EMISSIONS:")
    
    display_cols = [
        "Area",
        "Delta_Generation_TWh",
        "Generation_Change_%",
        "Delta_Capacity_GW",
        "Delta_Emissions_MtCO2e",
        "Emissions_Change_%",
        "Delta_Emissions_Intensity",
        "Delta_Share_Emissions_pp",
    ]

    print(
        generation_up_both_down
        .sort_values("Delta_Emissions_MtCO2e")
        [display_cols]
        .head(15)
        .round(3)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 5. Pattern ngược lại:
#    Generation tăng + emissions tăng + intensity tăng
# ------------------------------------------------------------
generation_up_carbon_worse = carbon_features[
    (carbon_features["Delta_Generation_TWh"] > 0) &
    (carbon_features["Delta_Emissions_MtCO2e"] > 0) &
    (carbon_features["Delta_Emissions_Intensity"] > 0)
].copy()

print("\n" + "=" * 80)
print("PATTERN: GENERATION TĂNG + CARBON XẤU ĐI")
print("=" * 80)

print(
    f"\nGeneration tăng + Emissions tăng + "
    f"Carbon intensity tăng: "
    f"{len(generation_up_carbon_worse)} quốc gia"
)

if len(generation_up_carbon_worse) > 0:
    print("\nTOP 15 QUỐC GIA THEO MỨC TĂNG EMISSIONS:")

    print(
        generation_up_carbon_worse
        .sort_values("Delta_Emissions_MtCO2e", ascending=False)
        [display_cols]
        .head(15)
        .round(3)
        .to_string(index=False)
    )

print(
    "→ Xác định phân phối và các pattern Carbon Transition trong 209 quốc gia."
)
print(
    "→ Tiếp theo sẽ kiểm tra mối quan hệ giữa cơ cấu nguồn điện và Carbon Transition."
)

PHÂN PHỐI CARBON + CAPACITY TRANSITION

THỐNG KÊ MÔ TẢ:
                           count    mean      std      min     25%     50%     75%       max  missing
Delta_Generation_TWh       209.0  30.537  270.080  -99.220   0.000   0.650   7.100  3822.177        0
Generation_Change_%        209.0  49.870  151.324  -73.451   0.000  23.056  54.954  1880.000        0
Delta_Capacity_GW          209.0  13.810   94.184   -7.880   0.100   0.690   4.570  1330.600        0
Capacity_Change_%          206.0  68.843   88.129  -15.105  17.205  39.667  84.493   638.889        3
Delta_Emissions_MtCO2e     209.0   8.805  120.168 -393.666  -0.341   0.033   1.377  1599.168        0
Emissions_Change_%         209.0  68.616  323.053  -86.957 -15.725  10.673  49.170  3844.625        0
Delta_Emissions_Intensity  209.0 -41.673   92.217 -326.923 -89.182 -30.598   0.000   350.000        0
Delta_Share_Emissions_pp   209.0  -0.041    1.002   -8.508  -0.014   0.000   0.010     8.556        0

HƯỚNG THAY ĐỔI 2012 → 202

In [6]:
# ============================================================
# ENERGY MIX ↔ CARBON TRANSITION
#
# - Fossil giảm có đi cùng emissions giảm không?
# - Renewables tăng có đi cùng carbon intensity giảm không?
# - Solar / Wind / Hydro / Nuclear có quan hệ khác nhau với Carbon Transition hay không?
#
# Mục đích: Kết nối kết quả Note 06 (Energy Transition) với Note 08
# (Carbon + Capacity Transition), từ đó xác định những biến đáng phân tích sâu hơn.
#
# Lưu ý:
# - Không dùng Delta_Share_Emissions_pp vì biến này gần như không thay đổi ở dòng Total generation.
# - Không diễn giải correlation là quan hệ nhân quả.
# ============================================================

from scipy.stats import pearsonr, spearmanr

# ------------------------------------------------------------
# 1. Ghép Energy Transition của Note 06 với Carbon Transition
# ------------------------------------------------------------
relationship_df = (
    transition_features
    .merge(
        carbon_features,
        on="Area",
        how="inner"
    )
)

print("=" * 80)
print("ENERGY MIX ↔ CARBON TRANSITION")
print("=" * 80)

print(
    f"\nCountries sau khi ghép: "
    f"{relationship_df['Area'].nunique()}"
)

print(
    f"Shape: {relationship_df.shape}"
)

# ------------------------------------------------------------
# 2. Các biến Energy Mix từ Note 06
# ------------------------------------------------------------
mix_variables = {
    "Fossil": "Delta_Fossil_share_pp",
    "Renewables": "Delta_Renewables_share_pp",
    "Solar": "Delta_Solar_share_pp",
    "Wind": "Delta_Wind_share_pp",
    "Hydro": "Delta_Hydro_share_pp",
    "Nuclear": "Delta_Nuclear_share_pp",
}

# ------------------------------------------------------------
# 3. Các biến Carbon Transition thực sự có ý nghĩa
# ------------------------------------------------------------
carbon_targets = {
    "Emissions": "Delta_Emissions_MtCO2e",
    "Emissions_%": "Emissions_Change_%",
    "Carbon_intensity": "Delta_Emissions_Intensity",
    "Generation": "Delta_Generation_TWh",
    "Generation_%": "Generation_Change_%",
    "Capacity": "Delta_Capacity_GW",
    "Capacity_%": "Capacity_Change_%",
}

# ------------------------------------------------------------
# 4. Pearson + Spearman
# ------------------------------------------------------------
results = []

for mix_name, mix_col in mix_variables.items():

    for target_name, target_col in carbon_targets.items():

        temp = relationship_df[
            [mix_col, target_col]
        ].dropna()

        if len(temp) < 3:
            continue

        pearson_r, pearson_p = pearsonr(
            temp[mix_col],
            temp[target_col]
        )

        spearman_r, spearman_p = spearmanr(
            temp[mix_col],
            temp[target_col]
        )

        results.append({
            "Energy_mix": mix_name,
            "Carbon_target": target_name,
            "N": len(temp),
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_rho": spearman_r,
            "Spearman_p": spearman_p,
        })

relationship_results = pd.DataFrame(results)

# ------------------------------------------------------------
# 5. Hiển thị toàn bộ kết quả
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("PEARSON + SPEARMAN")
print("=" * 80)

print(
    relationship_results
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Xếp hạng các mối quan hệ với Carbon Intensity
# ------------------------------------------------------------
intensity_results = (
    relationship_results[
        relationship_results["Carbon_target"]
        == "Carbon_intensity"
    ]
    .copy()
)

intensity_results["abs_Spearman"] = (
    intensity_results["Spearman_rho"].abs()
)

intensity_results = (
    intensity_results
    .sort_values("abs_Spearman", ascending=False)
)

print("\n" + "=" * 80)
print("RANKING — ENERGY MIX ↔ CARBON INTENSITY")
print("=" * 80)

print(
    intensity_results[
        [
            "Energy_mix",
            "N",
            "Pearson_r",
            "Pearson_p",
            "Spearman_rho",
            "Spearman_p",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. Xếp hạng các mối quan hệ với Emissions
# ------------------------------------------------------------
emissions_results = (
    relationship_results[
        relationship_results["Carbon_target"]
        == "Emissions"
    ]
    .copy()
)

emissions_results["abs_Spearman"] = (
    emissions_results["Spearman_rho"].abs()
)

emissions_results = (
    emissions_results
    .sort_values("abs_Spearman", ascending=False)
)

print("\n" + "=" * 80)
print("RANKING — ENERGY MIX ↔ EMISSIONS")
print("=" * 80)

print(
    emissions_results[
        [
            "Energy_mix",
            "N",
            "Pearson_r",
            "Pearson_p",
            "Spearman_rho",
            "Spearman_p",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


ENERGY MIX ↔ CARBON TRANSITION

Countries sau khi ghép: 209
Shape: (209, 15)

PEARSON + SPEARMAN
Energy_mix    Carbon_target   N  Pearson_r  Pearson_p  Spearman_rho  Spearman_p
    Fossil        Emissions 209     0.0119     0.8641        0.4442      0.0000
    Fossil      Emissions_% 209     0.3303     0.0000        0.5494      0.0000
    Fossil Carbon_intensity 209     0.8951     0.0000        0.9271      0.0000
    Fossil       Generation 209    -0.0196     0.7777        0.1210      0.0811
    Fossil     Generation_% 209     0.1004     0.1479        0.2016      0.0034
    Fossil         Capacity 209    -0.0382     0.5832       -0.1139      0.1007
    Fossil       Capacity_% 206    -0.1171     0.0937       -0.1042      0.1362
Renewables        Emissions 209    -0.0278     0.6892       -0.4791      0.0000
Renewables      Emissions_% 209    -0.3299     0.0000       -0.5606      0.0000
Renewables Carbon_intensity 209    -0.8865     0.0000       -0.9056      0.0000
Renewables       Genera

In [7]:
# ============================================================
# SO SÁNH CÁC CARBON TRANSITION PATTERNS
#
# Các quốc gia có kết quả Carbon Transition tốt và xấu
# có cơ cấu chuyển dịch nguồn điện khác nhau như thế nào?
#
# Sub-question:
# So sánh hai nhóm:
# A. Generation tăng + Emissions giảm + Carbon intensity giảm
# B. Generation tăng + Emissions tăng + Carbon intensity tăng
#
# Mục đích:
# - Không chỉ nhìn correlation toàn bộ 209 quốc gia.
# - Kiểm tra sự khác biệt về Energy Mix Transition giữa hai kiểu Carbon Transition đối lập.
# - Xác định nguồn điện nào nổi bật trong từng pattern.
# ============================================================

# ------------------------------------------------------------
# 1. Tạo nhãn Carbon Transition Pattern
# ------------------------------------------------------------

comparison_df = relationship_df.copy()

comparison_df["Carbon_Transition_Pattern"] = "Other"

# Nhóm A: generation tăng + emissions giảm + intensity giảm
comparison_df.loc[
    (comparison_df["Delta_Generation_TWh"] > 0) &
    (comparison_df["Delta_Emissions_MtCO2e"] < 0) &
    (comparison_df["Delta_Emissions_Intensity"] < 0),
    "Carbon_Transition_Pattern"
] = "Generation↑ / Carbon↓"

# Nhóm B: generation tăng + emissions tăng + intensity tăng
comparison_df.loc[
    (comparison_df["Delta_Generation_TWh"] > 0) &
    (comparison_df["Delta_Emissions_MtCO2e"] > 0) &
    (comparison_df["Delta_Emissions_Intensity"] > 0),
    "Carbon_Transition_Pattern"
] = "Generation↑ / Carbon↑"

print("=" * 80)
print("SO SÁNH CARBON TRANSITION PATTERNS")
print("=" * 80)

print("\nSỐ QUỐC GIA THEO PATTERN:")
print(
    comparison_df["Carbon_Transition_Pattern"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 2. So sánh Energy Mix Transition giữa hai nhóm
# ------------------------------------------------------------

target_patterns = [
    "Generation↑ / Carbon↓",
    "Generation↑ / Carbon↑"
]

comparison_variables = [
    "Delta_Fossil_share_pp",
    "Delta_Renewables_share_pp",
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp",
    "Delta_Nuclear_share_pp",
]

group_summary = (
    comparison_df[
        comparison_df["Carbon_Transition_Pattern"]
        .isin(target_patterns)
    ]
    .groupby("Carbon_Transition_Pattern")[comparison_variables]
    .agg(["mean", "median"])
)

print("\n" + "=" * 80)
print("ENERGY MIX TRANSITION — MEAN + MEDIAN")
print("=" * 80)

print(
    group_summary
    .round(3)
    .to_string()
)

# ------------------------------------------------------------
# 3. Chênh lệch giữa hai nhóm
# ------------------------------------------------------------

good_group = comparison_df[
    comparison_df["Carbon_Transition_Pattern"]
    == "Generation↑ / Carbon↓"
]

bad_group = comparison_df[
    comparison_df["Carbon_Transition_Pattern"]
    == "Generation↑ / Carbon↑"
]

difference_rows = []

for col in comparison_variables:
    good_mean = good_group[col].mean()
    bad_mean = bad_group[col].mean()

    difference_rows.append({
        "Variable": col,
        "Good_pattern_mean": good_mean,
        "Bad_pattern_mean": bad_mean,
        "Difference_Good_minus_Bad": (
            good_mean - bad_mean
        )
    })

group_difference = pd.DataFrame(difference_rows)

print("\n" + "=" * 80)
print("CHÊNH LỆCH ENERGY MIX GIỮA HAI PATTERN")
print("=" * 80)

print(
    group_difference
    .sort_values(
        "Difference_Good_minus_Bad",
        ascending=False
    )
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 4. So sánh trực tiếp các chỉ số Carbon Transition
# ------------------------------------------------------------

carbon_comparison_variables = [
    "Delta_Generation_TWh",
    "Generation_Change_%",
    "Delta_Capacity_GW",
    "Capacity_Change_%",
    "Delta_Emissions_MtCO2e",
    "Emissions_Change_%",
    "Delta_Emissions_Intensity",
]

carbon_group_summary = (
    comparison_df[
        comparison_df["Carbon_Transition_Pattern"]
        .isin(target_patterns)
    ]
    .groupby("Carbon_Transition_Pattern")
    [carbon_comparison_variables]
    .agg(["mean", "median"])
)

print("\n" + "=" * 80)
print("CARBON + CAPACITY TRANSITION — HAI NHÓM")
print("=" * 80)

print(
    carbon_group_summary
    .round(3)
    .to_string()
)

print(
    "→ Tiếp theo sẽ xác định nguồn điện nào thực sự phân biệt hai nhóm này."
)

SO SÁNH CARBON TRANSITION PATTERNS

SỐ QUỐC GIA THEO PATTERN:
Carbon_Transition_Pattern
Other                    131
Generation↑ / Carbon↑     41
Generation↑ / Carbon↓     37

ENERGY MIX TRANSITION — MEAN + MEDIAN
                          Delta_Fossil_share_pp         Delta_Renewables_share_pp         Delta_Solar_share_pp        Delta_Wind_share_pp        Delta_Hydro_share_pp        Delta_Nuclear_share_pp       
                                           mean  median                      mean  median                 mean median                mean median                 mean median                   mean median
Carbon_Transition_Pattern                                                                                                                                                                                 
Generation↑ / Carbon↑                     6.826   1.823                    -6.855  -1.823                1.529  0.391               0.343  0.000               -8.489 -4.704     

In [8]:
# ============================================================
# NGUỒN ĐIỆN PHÂN BIỆT HAI CARBON PATTERN
#
# Những nguồn điện nào có mức chuyển dịch khác biệt rõ nhất giữa nhóm Generation↑ / Carbon↓ và Generation↑ / Carbon↑?
#
# 1. Nguồn nào có effect size lớn nhất giữa hai nhóm?
# 2. Kết quả có còn rõ khi dùng median hay không?
# 3. Có bị chi phối bởi một vài quốc gia cực đoan không?
#
# Phương pháp:
# - Mann–Whitney U: so sánh phân phối giữa hai nhóm.
# - Cohen's d: đo độ lớn khác biệt theo mean/std.
# - Median difference: kiểm tra robustness với extreme values.
# ============================================================

from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# 1. Lấy hai nhóm
# ------------------------------------------------------------
good = comparison_df[
    comparison_df["Carbon_Transition_Pattern"]
    == "Generation↑ / Carbon↓"
].copy()

bad = comparison_df[
    comparison_df["Carbon_Transition_Pattern"]
    == "Generation↑ / Carbon↑"
].copy()

mix_variables_full = {
    "Fossil": "Delta_Fossil_share_pp",
    "Renewables": "Delta_Renewables_share_pp",
    "Solar": "Delta_Solar_share_pp",
    "Wind": "Delta_Wind_share_pp",
    "Hydro": "Delta_Hydro_share_pp",
    "Nuclear": "Delta_Nuclear_share_pp",
}

# ------------------------------------------------------------
# 2. Tính Mann–Whitney + Cohen's d + median difference
# ------------------------------------------------------------
results_88 = []

for source, col in mix_variables_full.items():

    x = good[col].dropna()
    y = bad[col].dropna()

    # Mann–Whitney U
    u_stat, p_value = mannwhitneyu(
        x,
        y,
        alternative="two-sided"
    )

    # Cohen's d
    mean_x = x.mean()
    mean_y = y.mean()

    std_x = x.std(ddof=1)
    std_y = y.std(ddof=1)

    pooled_std = np.sqrt(
        (
            (len(x) - 1) * std_x**2
            + (len(y) - 1) * std_y**2
        )
        / (len(x) + len(y) - 2)
    )

    cohens_d = (
        (mean_x - mean_y) / pooled_std
        if pooled_std != 0
        else np.nan
    )

    results_88.append({
        "Source": source,
        "Good_n": len(x),
        "Bad_n": len(y),
        "Good_mean": mean_x,
        "Bad_mean": mean_y,
        "Mean_difference": mean_x - mean_y,
        "Good_median": x.median(),
        "Bad_median": y.median(),
        "Median_difference": x.median() - y.median(),
        "Cohens_d": cohens_d,
        "MW_U": u_stat,
        "MW_p": p_value,
    })

source_comparison = pd.DataFrame(results_88)

# ------------------------------------------------------------
# 3. Sắp xếp theo effect size tuyệt đối
# ------------------------------------------------------------
source_comparison["abs_Cohens_d"] = (
    source_comparison["Cohens_d"].abs()
)

source_comparison = (
    source_comparison
    .sort_values("abs_Cohens_d", ascending=False)
)

print("=" * 80)
print("NGUỒN ĐIỆN PHÂN BIỆT HAI CARBON PATTERN")
print("=" * 80)

print(
    f"\nGood pattern: {len(good)} quốc gia"
)

print(
    f"Bad pattern: {len(bad)} quốc gia"
)

print("\n" + "=" * 80)
print("MANN–WHITNEY + COHEN'S d")
print("=" * 80)

print(
    source_comparison[
        [
            "Source",
            "Good_mean",
            "Bad_mean",
            "Mean_difference",
            "Good_median",
            "Bad_median",
            "Median_difference",
            "Cohens_d",
            "MW_p",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 4. Kiểm tra robustness: loại các extreme values theo 5% hai đầu của từng nguồn
# ------------------------------------------------------------
robust_results_88 = []

for source, col in mix_variables_full.items():

    combined = comparison_df[
        comparison_df["Carbon_Transition_Pattern"]
        .isin([
            "Generation↑ / Carbon↓",
            "Generation↑ / Carbon↑"
        ])
    ][[col, "Carbon_Transition_Pattern"]].dropna()

    lower = combined[col].quantile(0.05)
    upper = combined[col].quantile(0.95)

    robust = combined[
        (combined[col] >= lower) &
        (combined[col] <= upper)
    ]

    robust_good = robust[
        robust["Carbon_Transition_Pattern"]
        == "Generation↑ / Carbon↓"
    ][col]

    robust_bad = robust[
        robust["Carbon_Transition_Pattern"]
        == "Generation↑ / Carbon↑"
    ][col]

    if len(robust_good) > 1 and len(robust_bad) > 1:

        u, p = mannwhitneyu(
            robust_good,
            robust_bad,
            alternative="two-sided"
        )

        pooled_std = np.sqrt(
            (
                (len(robust_good) - 1) * robust_good.std(ddof=1)**2
                + (len(robust_bad) - 1) * robust_bad.std(ddof=1)**2
            )
            / (
                len(robust_good)
                + len(robust_bad)
                - 2
            )
        )

        d = (
            (robust_good.mean() - robust_bad.mean())
            / pooled_std
            if pooled_std != 0
            else np.nan
        )

        robust_results_88.append({
            "Source": source,
            "Good_n_robust": len(robust_good),
            "Bad_n_robust": len(robust_bad),
            "Robust_mean_difference": (
                robust_good.mean()
                - robust_bad.mean()
            ),
            "Robust_Cohens_d": d,
            "Robust_MW_p": p,
        })

robust_source_comparison = pd.DataFrame(
    robust_results_88
)

print("\n" + "=" * 80)
print("ROBUSTNESS — LOẠI 5% EXTREME VALUES")
print("=" * 80)

print(
    robust_source_comparison
    .sort_values(
        "Robust_Cohens_d",
        key=lambda s: s.abs(),
        ascending=False
    )
    .round(4)
    .to_string(index=False)
)

print(
    "→ Đã xác định effect size và significance của từng nguồn."
)
print(
    "→ Đã kiểm tra robustness sau khi loại 5% extreme values."
)
print(
    "→ Chưa chọn driver cuối cùng; lựa chọn sẽ dựa trên effect size + median + robustness."
)

NGUỒN ĐIỆN PHÂN BIỆT HAI CARBON PATTERN

Good pattern: 37 quốc gia
Bad pattern: 41 quốc gia

MANN–WHITNEY + COHEN'S d
    Source  Good_mean  Bad_mean  Mean_difference  Good_median  Bad_median  Median_difference  Cohens_d   MW_p
Renewables    15.3383   -6.8554          22.1937      13.0040     -1.8230            14.8270    2.1232 0.0000
    Fossil   -14.9379    6.8260         -21.7639     -11.4810      1.8230           -13.3040   -2.0483 0.0000
      Wind     6.6878    0.3434           6.3444       4.7825      0.0000             4.7825    1.2159 0.0000
     Hydro     0.8487   -8.4893           9.3381      -0.0370     -4.7035             4.6665    0.9320 0.0001
     Solar     6.3679    1.5293           4.8386       4.6530      0.3910             4.2620    0.7481 0.0000
   Nuclear    -0.4232    0.0309          -0.4541       0.0000      0.0000             0.0000   -0.4064 0.0611

ROBUSTNESS — LOẠI 5% EXTREME VALUES
    Source  Good_n_robust  Bad_n_robust  Robust_mean_difference  Robust_Coh

In [9]:
# ============================================================
# - Có năm nào Solar/Wind tăng tỷ trọng bất thường không?
# - Trong từng Ember region, mức tăng Share of generation Solar/Wind ở 
# country-year nào vượt xa biến động thông thường của khu vực?
#
# Purpose:
#   - Khai thác trực tiếp biến "Share of generation YoY change (% points)"
#   - Tìm các annual transition events thay vì chỉ nhìn 2012 -> 2022
#   - Dùng Ember region làm baseline để tránh so sánh
#     các khu vực có đặc điểm năng lượng quá khác nhau.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# 1. Chỉ lấy Country/Economy + Solar/Wind và giai đoạn 2012–2022 
# vì đây là giai đoạn có coverage quốc gia ổn định để so sánh.
# ------------------------------------------------------------
anomaly_df = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"].isin(["Solar", "Wind"])) &
    (df["Year"].between(2012, 2022))
].copy()

print("\n✓ Filtered analysis data")
print(f"  Rows: {len(anomaly_df):,}")
print(f"  Countries: {anomaly_df['Area'].nunique()}")
print(f"  Sources: {anomaly_df['Electricity source'].unique().tolist()}")
print(f"  Years: {anomaly_df['Year'].min()}–{anomaly_df['Year'].max()}")

# ------------------------------------------------------------
# 2. Kiểm tra coverage trước khi tính anomaly
# ------------------------------------------------------------
coverage = (
    anomaly_df
    .groupby(["Electricity source", "Year"])["Area"]
    .nunique()
    .reset_index(name="countries")
)

print("\n--- Country coverage by source/year ---")
print(coverage.to_string(index=False))

# ------------------------------------------------------------
# 3. Tính baseline theo Ember region + source
#
#    z = (country-year change - regional mean)
#        / regional std
#
#    |z| >= 2.5 được xem là candidate anomaly.
# ------------------------------------------------------------
baseline = (
    anomaly_df
    .groupby(["Ember region", "Electricity source"])
    ["Share of generation YoY change (% points)"]
    .agg(
        baseline_mean="mean",
        baseline_std="std",
        baseline_n="count"
    )
    .reset_index()
)

anomaly_df = anomaly_df.merge(
    baseline,
    on=["Ember region", "Electricity source"],
    how="left"
)

anomaly_df["z_score_region"] = (
    anomaly_df["Share of generation YoY change (% points)"]
    - anomaly_df["baseline_mean"]
) / anomaly_df["baseline_std"]

# ------------------------------------------------------------
# 4. Candidate anomalies
#    Chỉ giữ baseline đủ quan sát để std có ý nghĩa.
# ------------------------------------------------------------
anomalies = anomaly_df[
    (anomaly_df["baseline_n"] >= 30) &
    (anomaly_df["z_score_region"].abs() >= 2.5)
].copy()

anomalies["abs_z"] = anomalies["z_score_region"].abs()

anomalies = anomalies.sort_values(
    "abs_z",
    ascending=False
)

print("\n" + "=" * 80)
print("CANDIDATE SOLAR/WIND SHARE ANOMALIES")
print("=" * 80)

print(
    anomalies[
        [
            "Area",
            "Year",
            "Ember region",
            "Electricity source",
            "Share of generation YoY change (% points)",
            "baseline_mean",
            "baseline_std",
            "z_score_region"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

print(f"\n✓ Total candidate anomalies: {len(anomalies)}")


✓ Filtered analysis data
  Rows: 4,547
  Countries: 209
  Sources: ['Solar', 'Wind']
  Years: 2012–2022

--- Country coverage by source/year ---
Electricity source  Year  countries
             Solar  2012        209
             Solar  2013        209
             Solar  2014        209
             Solar  2015        209
             Solar  2016        209
             Solar  2017        209
             Solar  2018        209
             Solar  2019        209
             Solar  2020        209
             Solar  2021        209
             Solar  2022        209
              Wind  2012        201
              Wind  2013        202
              Wind  2014        204
              Wind  2015        204
              Wind  2016        204
              Wind  2017        205
              Wind  2018        205
              Wind  2019        205
              Wind  2020        206
              Wind  2021        206
              Wind  2022        206

CANDIDATE SOLAR/WIND SHAR

In [10]:
# ============================================================
# Các anomaly về Solar/Wind share có phải là những sự kiện chuyển dịch năng lượng thực sự?
# Khi Share tăng/giảm bất thường, Generation của nguồn đó và Total generation thay đổi như thế nào?
#
#   Phân biệt:
#   1. True expansion: source generation tăng mạnh
#   2. Share anomaly do denominator effect
#   3. Structural transition: share tăng + fossil giảm
# ============================================================

# ------------------------------------------------------------
# 1. Lấy các candidate anomaly mạnh nhất
# ------------------------------------------------------------
top_events = anomalies.head(20).copy()

# ------------------------------------------------------------
# 2. Lấy dữ liệu năm trước và năm xảy ra anomaly
# ------------------------------------------------------------
event_keys = top_events[
    ["Area", "Year", "Electricity source"]
].copy()

current = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"].isin(["Solar", "Wind"]))
].copy()

previous = current.copy()
previous["Year"] = previous["Year"] + 1

event_compare = event_keys.merge(
    current[
        [
            "Area",
            "Year",
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)"
        ]
    ].rename(columns={
        "Generation (TWh)": "Generation_current_TWh",
        "Share of generation (%)": "Share_current_pct"
    }),
    on=["Area", "Year", "Electricity source"],
    how="left"
)

event_compare = event_compare.merge(
    previous[
        [
            "Area",
            "Year",
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)"
        ]
    ].rename(columns={
        "Generation (TWh)": "Generation_previous_TWh",
        "Share of generation (%)": "Share_previous_pct"
    }),
    on=["Area", "Year", "Electricity source"],
    how="left"
)

# ------------------------------------------------------------
# 3. Thêm Total generation của năm hiện tại và năm trước
# ------------------------------------------------------------
total = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"] == "Total generation")
][
    ["Area", "Year", "Generation (TWh)"]
].copy()

event_compare = event_compare.merge(
    total.rename(columns={
        "Generation (TWh)": "Total_generation_current_TWh"
    }),
    on=["Area", "Year"],
    how="left"
)

total_prev = total.copy()
total_prev["Year"] = total_prev["Year"] + 1

event_compare = event_compare.merge(
    total_prev.rename(columns={
        "Generation (TWh)": "Total_generation_previous_TWh"
    }),
    on=["Area", "Year"],
    how="left"
)

# ------------------------------------------------------------
# 4. Tính mức thay đổi
# ------------------------------------------------------------
event_compare["Generation_change_TWh"] = (
    event_compare["Generation_current_TWh"]
    - event_compare["Generation_previous_TWh"]
)

event_compare["Total_generation_change_TWh"] = (
    event_compare["Total_generation_current_TWh"]
    - event_compare["Total_generation_previous_TWh"]
)

event_compare["Share_change_pp"] = (
    event_compare["Share_current_pct"]
    - event_compare["Share_previous_pct"]
)

event_compare["Generation_growth_pct"] = (
    event_compare["Generation_change_TWh"]
    / event_compare["Generation_previous_TWh"].replace(0, np.nan)
    * 100
)

# ------------------------------------------------------------
# 5. Hiển thị
# ------------------------------------------------------------
result = event_compare.merge(
    top_events[
        [
            "Area",
            "Year",
            "Electricity source",
            "Ember region",
            "z_score_region"
        ]
    ],
    on=["Area", "Year", "Electricity source"],
    how="left"
)

result = result.sort_values(
    "z_score_region",
    key=lambda x: x.abs(),
    ascending=False
)

print("=" * 100)
print("VALIDATION OF TOP SOLAR/WIND SHARE ANOMALIES")
print("=" * 100)

print(
    result[
        [
            "Area",
            "Year",
            "Electricity source",
            "Ember region",
            "z_score_region",
            "Generation_previous_TWh",
            "Generation_current_TWh",
            "Generation_change_TWh",
            "Generation_growth_pct",
            "Total_generation_previous_TWh",
            "Total_generation_current_TWh",
            "Total_generation_change_TWh",
            "Share_previous_pct",
            "Share_current_pct",
            "Share_change_pp"
        ]
    ].to_string(index=False)
)

VALIDATION OF TOP SOLAR/WIND SHARE ANOMALIES
                       Area  Year Electricity source                Ember region  z_score_region  Generation_previous_TWh  Generation_current_TWh  Generation_change_TWh  Generation_growth_pct  Total_generation_previous_TWh  Total_generation_current_TWh  Total_generation_change_TWh  Share_previous_pct  Share_current_pct  Share_change_pp
                   Djibouti  2021               Wind                      Africa       20.978063                     0.00                    0.07                   0.07                    NaN                           0.13                          0.20                         0.07               0.000             35.000           35.000
Falkland Islands (Malvinas)  2017               Wind Latin America and Caribbean      -19.006283                     0.01                    0.00                  -0.01            -100.000000                           0.02                          0.01                        -0.

In [11]:
# ============================================================
# Trong các Solar/Wind anomalies, event nào thực sự phản ánh sự mở rộng/thu hẹp của nguồn điện?
#
#   Candidate anomaly có đồng thời:
#   (1) share thay đổi đáng kể,
#   (2) generation nguồn thay đổi cùng chiều,
#   (3) total generation không bị sụt mạnh?
#
# Loại các anomaly chủ yếu do denominator effect hoặc quy mô hệ thống quá nhỏ, 
# để tạo tập "robust events" dùng cho phân tích sâu và visualization.
# ============================================================

validated = result.copy()

# 1. Xác định hướng của share và generation
validated["share_direction"] = np.sign(validated["Share_change_pp"])
validated["generation_direction"] = np.sign(
    validated["Generation_change_TWh"]
)

# 2. Generation và share cùng chiều
validated["same_direction"] = (
    validated["share_direction"]
    == validated["generation_direction"]
)

# 3. Mức thay đổi Total generation theo %
validated["Total_generation_change_pct"] = (
    validated["Total_generation_change_TWh"]
    / validated["Total_generation_previous_TWh"].replace(0, np.nan)
    * 100
)

# 4. Phân loại event
def classify_event(row):
    if not row["same_direction"]:
        return "Denominator-driven"

    # Nguồn thay đổi cùng chiều với share
    if row["Share_change_pp"] > 0:
        if pd.notna(row["Total_generation_change_pct"]) and row["Total_generation_change_pct"] < -20:
            return "Expansion + total generation decline"
        return "True expansion"

    if row["Share_change_pp"] < 0:
        if pd.notna(row["Total_generation_change_pct"]) and row["Total_generation_change_pct"] > 20:
            return "Decline + total generation growth"
        return "True decline"

    return "Other"

validated["event_type"] = validated.apply(classify_event, axis=1)

# 5. Chỉ giữ các event có quy mô nguồn đáng kể hơn
#    Không dùng ngưỡng tuyệt đối để loại ngay; chỉ tạo cột
#    để nhìn quy mô trước khi quyết định threshold cuối cùng.
validated["source_generation_abs_TWh"] = (
    validated["Generation_change_TWh"].abs()
)

print("=" * 100)
print("CLASSIFICATION OF SOLAR/WIND ANOMALIES")
print("=" * 100)

print("\nSố lượng theo loại event:")
print(
    validated["event_type"]
    .value_counts()
    .to_string()
)

print("\nTop TRUE EXPANSION:")
print(
    validated[
        validated["event_type"] == "True expansion"
    ][
        [
            "Area",
            "Year",
            "Electricity source",
            "z_score_region",
            "Generation_previous_TWh",
            "Generation_current_TWh",
            "Generation_change_TWh",
            "Generation_growth_pct",
            "Total_generation_change_TWh",
            "Total_generation_change_pct",
            "Share_previous_pct",
            "Share_current_pct",
            "Share_change_pp"
        ]
    ]
    .sort_values(
        "Share_change_pp",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

print("\nTop DENOMINATOR-DRIVEN:")
print(
    validated[
        validated["event_type"] == "Denominator-driven"
    ][
        [
            "Area",
            "Year",
            "Electricity source",
            "z_score_region",
            "Generation_change_TWh",
            "Total_generation_change_TWh",
            "Total_generation_change_pct",
            "Share_change_pp"
        ]
    ]
    .sort_values(
        "z_score_region",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)

CLASSIFICATION OF SOLAR/WIND ANOMALIES

Số lượng theo loại event:
event_type
True expansion                          13
Expansion + total generation decline     4
True decline                             3

Top TRUE EXPANSION:
       Area  Year Electricity source  z_score_region  Generation_previous_TWh  Generation_current_TWh  Generation_change_TWh  Generation_growth_pct  Total_generation_change_TWh  Total_generation_change_pct  Share_previous_pct  Share_current_pct  Share_change_pp
   Djibouti  2021               Wind       20.978063                     0.00                    0.07                   0.07                    NaN                         0.07                    53.846154               0.000             35.000           35.000
    Vanuatu  2013               Wind        6.974098                     0.00                    0.01                   0.01                    NaN                         0.01                    16.666667               0.000             14.286     

In [12]:
# ============================================================
# Các Solar/Wind anomaly có quy mô đủ lớn để xem là một sự kiện chuyển dịch đáng kể của hệ thống điện?
#
#   Candidate anomalies phân bố thế nào theo:
#   - Generation của Solar/Wind
#   - Total generation của hệ thống
#   - Absolute change của Solar/Wind
#
# Quan sát phân phối để phân biệt:
#   - Không tự đặt threshold trước.
#   - anomaly có quy mô hệ thống đáng kể
#   - anomaly do hệ thống điện rất nhỏ
# ============================================================

# validated đã chứa:
# - Generation_current_TWh
# - Generation_change_TWh
# - Total_generation_current_TWh
# - Total_generation_change_TWh
# - Total_generation_change_pct
# - Share_change_pp
# - z_score_region

scale_check = validated.copy()

print("=" * 100)
print("SCALE DISTRIBUTION OF SOLAR/WIND ANOMALIES")
print("=" * 100)

print("\nSố candidate anomalies:", len(scale_check))

print("\n--- Generation hiện tại của Solar/Wind ---")
print(
    scale_check["Generation_current_TWh"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .to_string()
)

print("\n--- Total generation hiện tại ---")
print(
    scale_check["Total_generation_current_TWh"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .to_string()
)

print("\n--- Absolute change của Solar/Wind ---")
print(
    scale_check["Generation_change_TWh"]
    .abs()
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .to_string()
)

print("\n--- Top 20 theo absolute generation change ---")
print(
    scale_check[
        [
            "Area",
            "Year",
            "Electricity source",
            "z_score_region",
            "Generation_change_TWh",
            "Generation_current_TWh",
            "Total_generation_current_TWh",
            "Share_change_pp"
        ]
    ]
    .sort_values(
        "Generation_change_TWh",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

SCALE DISTRIBUTION OF SOLAR/WIND ANOMALIES

Số candidate anomalies: 20

--- Generation hiện tại của Solar/Wind ---
count    20.000000
mean      2.213000
std       5.974551
min       0.000000
25%       0.070000
50%       0.330000
75%       1.245000
90%       2.430000
95%       9.943000
99%      22.908600
max      26.150000

--- Total generation hiện tại ---
count     20.0000
mean      29.2525
std       78.8638
min        0.0100
25%        0.4725
50%        1.4600
75%        6.7075
90%       43.9910
95%      254.1850
99%      262.9250
max      265.1100

--- Absolute change của Solar/Wind ---
count    20.000000
mean      1.257500
std       3.557185
min       0.010000
25%       0.055000
50%       0.125000
75%       0.462500
90%       1.430000
95%       6.232000
99%      13.558400
max      15.390000

--- Top 20 theo absolute generation change ---
                       Area  Year Electricity source  z_score_region  Generation_change_TWh  Generation_current_TWh  Total_generation_current_TWh 

In [13]:
# ============================================================
# Tập hợp các Solar/Wind anomalies nào đủ cơ sở để trở thành "transition events" chính thức của Note 08?
#
#   Event có đồng thời thỏa:
#   - anomaly thống kê mạnh
#   - share thay đổi đáng kể
#   - generation thay đổi cùng hướng
#
#   Tạo event table chuẩn để dùng cho:
#   1. ranking các sự kiện
#   2. visualization
#   3. phân tích decoupling tiếp theo
#
# Lưu ý: Không loại các hệ thống nhỏ khỏi dữ liệu gốc. Chỉ gắn nhãn quy mô để tránh diễn giải sai.
# ============================================================

events = validated.copy()

# 1. Cường độ anomaly
events["anomaly_strength"] = events["z_score_region"].abs()

# 2. Quy mô hệ thống
events["system_scale"] = pd.cut(
    events["Total_generation_current_TWh"],
    bins=[-np.inf, 1, 10, 100, np.inf],
    labels=[
        "Very small (<1 TWh)",
        "Small (1-10 TWh)",
        "Medium (10-100 TWh)",
        "Large (>100 TWh)"
    ]
)

# 3. Quy mô thay đổi nguồn
events["source_change_abs_TWh"] = (
    events["Generation_change_TWh"].abs()
)

# 4. Event hợp lệ về mặt hướng
events["robust_direction"] = (
    events["same_direction"]
    & (events["Share_change_pp"].abs() >= 2)
)

# 5. Phân loại event cuối
def final_event_class(row):
    if not row["robust_direction"]:
        return "Weak / denominator-sensitive"

    if row["Share_change_pp"] > 0:
        if (
            pd.notna(row["Total_generation_change_pct"])
            and row["Total_generation_change_pct"] < -20
        ):
            return "Renewable expansion + system contraction"
        return "Renewable expansion"

    return "Renewable decline"

events["final_event_class"] = events.apply(
    final_event_class,
    axis=1
)

# 6. Bảng event chính thức
event_table = events[
    events["robust_direction"]
].copy()

event_table = event_table.sort_values(
    ["source_change_abs_TWh", "anomaly_strength"],
    ascending=[False, False]
)

print("=" * 110)
print("FINAL SOLAR/WIND TRANSITION EVENT TABLE")
print("=" * 110)

print("\nSố event sau validation:", len(event_table))

print("\nPhân loại event:")
print(
    event_table["final_event_class"]
    .value_counts()
    .to_string()
)

print("\nQuy mô hệ thống:")
print(
    event_table["system_scale"]
    .value_counts()
    .to_string()
)

print("\nTOP EVENTS:")
print(
    event_table[
        [
            "Area",
            "Year",
            "Electricity source",
            "final_event_class",
            "system_scale",
            "anomaly_strength",
            "Generation_change_TWh",
            "Generation_growth_pct",
            "Total_generation_change_pct",
            "Share_change_pp"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


FINAL SOLAR/WIND TRANSITION EVENT TABLE

Số event sau validation: 20

Phân loại event:
final_event_class
Renewable expansion                         13
Renewable expansion + system contraction     4
Renewable decline                            3

Quy mô hệ thống:
system_scale
Small (1-10 TWh)       10
Very small (<1 TWh)     7
Large (>100 TWh)        2
Medium (10-100 TWh)     1

TOP EVENTS:
                       Area  Year Electricity source                        final_event_class        system_scale  anomaly_strength  Generation_change_TWh  Generation_growth_pct  Total_generation_change_pct  Share_change_pp
                   Viet Nam  2021              Solar                      Renewable expansion    Large (>100 TWh)         10.199120                  15.39             143.029740                     3.938525            5.901
                   Viet Nam  2022               Wind                      Renewable expansion    Large (>100 TWh)          6.350969                   5.75   

In [14]:
# ============================================================
# Sản lượng điện và phát thải có luôn thay đổi cùng chiều?
#
#   Trong các country-year, có trường hợp:
#   - Generation tăng nhưng Emissions giảm?
#   - Generation giảm nhưng Emissions tăng?
#
# Có phát hiện các "decoupling events" — những năm tăng trưởng điện và phát thải đi ngược chiều?
#
# Đây là bước chuyển từ mô tả energy transition sang phát hiện những thay đổi bất thường có ý nghĩa môi trường.
# ============================================================

# 1. Lấy Total generation và emissions cho country/economy
decoupling = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"] == "Total generation")
][
    [
        "Area",
        "Year",
        "Generation (TWh)",
        "Generation YoY change (TWh)",
        "Generation YoY change (%)",
        "Emissions (MtCO2e)",
        "Emissions YoY change (MtCO2e)",
        "Emissions YoY change (%)",
        "Emissions intensity (gCO2e/kWh)",
        "Ember region"
    ]
].copy()

# 2. Chỉ giữ các quan sát có đủ hai biến thay đổi
decoupling = decoupling.dropna(
    subset=[
        "Generation YoY change (TWh)",
        "Emissions YoY change (MtCO2e)"
    ]
).copy()

# 3. Phân loại hướng thay đổi
def classify_decoupling(row):
    gen_change = row["Generation YoY change (TWh)"]
    emission_change = row["Emissions YoY change (MtCO2e)"]

    if gen_change > 0 and emission_change < 0:
        return "Generation ↑ / Emissions ↓"

    if gen_change < 0 and emission_change > 0:
        return "Generation ↓ / Emissions ↑"

    if gen_change > 0 and emission_change > 0:
        return "Generation ↑ / Emissions ↑"

    if gen_change < 0 and emission_change < 0:
        return "Generation ↓ / Emissions ↓"

    return "No material change"

decoupling["decoupling_type"] = decoupling.apply(
    classify_decoupling,
    axis=1
)

print("=" * 110)
print("GENERATION–EMISSIONS DECOUPLING ANALYSIS")
print("=" * 110)

print("\nSố country-year observations:", len(decoupling))

print("\nPhân bố các kiểu thay đổi:")
print(
    decoupling["decoupling_type"]
    .value_counts()
    .to_string()
)

print("\n--- Generation ↑ / Emissions ↓ ---")
print(
    decoupling[
        decoupling["decoupling_type"]
        == "Generation ↑ / Emissions ↓"
    ][
        [
            "Area",
            "Year",
            "Ember region",
            "Generation (TWh)",
            "Generation YoY change (TWh)",
            "Generation YoY change (%)",
            "Emissions (MtCO2e)",
            "Emissions YoY change (MtCO2e)",
            "Emissions YoY change (%)",
            "Emissions intensity (gCO2e/kWh)"
        ]
    ]
    .sort_values(
        "Emissions YoY change (MtCO2e)"
    )
    .head(20)
    .to_string(index=False)
)

print("\n--- Generation ↓ / Emissions ↑ ---")
print(
    decoupling[
        decoupling["decoupling_type"]
        == "Generation ↓ / Emissions ↑"
    ][
        [
            "Area",
            "Year",
            "Ember region",
            "Generation (TWh)",
            "Generation YoY change (TWh)",
            "Generation YoY change (%)",
            "Emissions (MtCO2e)",
            "Emissions YoY change (MtCO2e)",
            "Emissions YoY change (%)",
            "Emissions intensity (gCO2e/kWh)"
        ]
    ]
    .sort_values(
        "Emissions YoY change (MtCO2e)",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

GENERATION–EMISSIONS DECOUPLING ANALYSIS

Số country-year observations: 5706

Phân bố các kiểu thay đổi:
decoupling_type
Generation ↑ / Emissions ↑    3096
Generation ↓ / Emissions ↓    1124
Generation ↑ / Emissions ↓     669
No material change             567
Generation ↓ / Emissions ↑     250

--- Generation ↑ / Emissions ↓ ---
          Area  Year                Ember region  Generation (TWh)  Generation YoY change (TWh)  Generation YoY change (%)  Emissions (MtCO2e)  Emissions YoY change (MtCO2e)  Emissions YoY change (%)  Emissions intensity (gCO2e/kWh)
         China  2015                        Asia          5798.818                       17.551                      0.304            3883.496                       -138.837                    -3.452                          669.705
 United States  2016               North America          4087.560                        3.570                      0.087            1975.898                        -86.289                    -4.184   

In [15]:
# ============================================================
# Trong 669 trường hợp Generation ↑ / Emissions ↓, trường hợp nào là decoupling mạnh và đáng chú ý nhất?
#
#   Có quốc gia nào vừa:
#   - tăng generation đáng kể,
#   - giảm emissions đáng kể,
#   - đồng thời giảm carbon intensity?
#
# Purpose: Tách "decoupling thật sự đáng chú ý" khỏi những thay đổi rất nhỏ về generation hoặc emissions.
#
# Không đặt threshold trước: Trước tiên xem phân phối của 669 observations, sau đó mới quyết định tiêu chí lọc.
# ============================================================

decoupling_up_down = decoupling[
    decoupling["decoupling_type"] == "Generation ↑ / Emissions ↓"
].copy()

# 1. Độ lớn tuyệt đối của thay đổi
decoupling_up_down["Generation_increase_abs"] = (
    decoupling_up_down["Generation YoY change (TWh)"]
)

decoupling_up_down["Emissions_reduction_abs"] = (
    -decoupling_up_down["Emissions YoY change (MtCO2e)"]
)

print("=" * 110)
print("STRENGTH OF GENERATION–EMISSIONS DECOUPLING")
print("=" * 110)

print("\nSố decoupling observations:", len(decoupling_up_down))

print("\n--- Generation increase ---")
print(
    decoupling_up_down[
        "Generation_increase_abs"
    ].describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    ).to_string()
)

print("\n--- Emissions reduction ---")
print(
    decoupling_up_down[
        "Emissions_reduction_abs"
    ].describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    ).to_string()
)

print("\n--- Generation growth (%) ---")
print(
    decoupling_up_down[
        "Generation YoY change (%)"
    ].describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    ).to_string()
)

print("\n--- Emissions change (%) ---")
print(
    decoupling_up_down[
        "Emissions YoY change (%)"
    ].describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    ).to_string()
)

print("\nTOP 20 THEO LƯỢNG GIẢM PHÁT THẢI:")
print(
    decoupling_up_down[
        [
            "Area",
            "Year",
            "Ember region",
            "Generation YoY change (TWh)",
            "Generation YoY change (%)",
            "Emissions YoY change (MtCO2e)",
            "Emissions YoY change (%)",
            "Emissions intensity (gCO2e/kWh)"
        ]
    ]
    .sort_values(
        "Emissions YoY change (MtCO2e)"
    )
    .head(20)
    .to_string(index=False)
)

print("\nTOP 20 THEO TĂNG GENERATION:")
print(
    decoupling_up_down[
        [
            "Area",
            "Year",
            "Ember region",
            "Generation YoY change (TWh)",
            "Generation YoY change (%)",
            "Emissions YoY change (MtCO2e)",
            "Emissions YoY change (%)",
            "Emissions intensity (gCO2e/kWh)"
        ]
    ]
    .sort_values(
        "Generation YoY change (TWh)",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


STRENGTH OF GENERATION–EMISSIONS DECOUPLING

Số decoupling observations: 669

--- Generation increase ---
count    669.000000
mean       3.665099
std       19.899899
min        0.010000
25%        0.180000
50%        0.740000
75%        2.700000
90%        7.894000
95%       13.486000
99%       28.833600
max      480.732000

--- Emissions reduction ---
count    669.000000
mean       2.079928
std        7.582460
min        0.001000
25%        0.063000
50%        0.367000
75%        1.453000
90%        4.485400
95%        8.456400
99%       21.698960
max      138.837000

--- Generation growth (%) ---
count    669.000000
mean       5.012178
std        8.241535
min        0.002000
25%        1.140000
50%        2.715000
75%        5.854000
90%       11.046800
95%       16.740800
99%       32.196280
max      133.333000

--- Emissions change (%) ---
count    669.000000
mean      -8.532132
std       13.773591
min     -255.634000
25%      -11.008000
50%       -4.853000
75%       -1.497000
90%

In [16]:
# ============================================================
# Trong 669 trường hợp Generation ↑ / Emissions ↓, nhóm nào thực sự có mức decoupling mạnh?
#
#   Có những country-year nào đồng thời:
#   1. tăng generation thuộc nhóm cao,
#   2. giảm emissions thuộc nhóm cao,
#   3. giảm carbon intensity?
#
# Purpose: Tạo một nhóm "strong decoupling" có cơ sở thống kê từ chính phân phối dữ liệu, thay vì đặt threshold tùy ý.
#
# P75 được dùng làm ngưỡng ban đầu vì:
#   - tránh giữ quá nhiều observation nhỏ;
#   - vẫn đủ rộng để phân tích nhóm;
#   - sau đó sẽ kiểm tra lại quy mô và intensity.
# ============================================================

strong_base = decoupling_up_down.copy()

# ------------------------------------------------------------
# 1. Tính ngưỡng P75 từ chính 669 observations
# ------------------------------------------------------------

gen_p75 = strong_base["Generation_increase_abs"].quantile(0.75)
emission_reduction_p75 = strong_base["Emissions_reduction_abs"].quantile(0.75)

print("=" * 110)
print("STRONG DECOUPLING — INITIAL SCREENING")
print("=" * 110)

print(f"\nP75 Generation increase: {gen_p75:.3f} TWh")
print(f"P75 Emissions reduction: {emission_reduction_p75:.3f} MtCO2e")

# ------------------------------------------------------------
# 2. Lọc nhóm có cả hai yếu tố đều thuộc top 25%
# ------------------------------------------------------------

strong_base["high_generation_increase"] = (
    strong_base["Generation_increase_abs"] >= gen_p75
)

strong_base["high_emissions_reduction"] = (
    strong_base["Emissions_reduction_abs"] >= emission_reduction_p75
)

strong_decoupling = strong_base[
    strong_base["high_generation_increase"]
    & strong_base["high_emissions_reduction"]
].copy()

print("\nSố observations đạt cả 2 điều kiện:",
      len(strong_decoupling))

print(
    f"Tỷ lệ trên 669 decoupling observations: "
    f"{len(strong_decoupling) / len(strong_base) * 100:.2f}%"
)

# ------------------------------------------------------------
# 3. Xem intensity của nhóm strong decoupling
# ------------------------------------------------------------

print("\n--- STRONG DECOUPLING ---")

print(
    strong_decoupling[
        [
            "Area",
            "Year",
            "Ember region",
            "Generation_increase_abs",
            "Generation YoY change (%)",
            "Emissions_reduction_abs",
            "Emissions YoY change (%)",
            "Emissions intensity (gCO2e/kWh)"
        ]
    ]
    .sort_values(
        "Emissions_reduction_abs",
        ascending=False
    )
    .to_string(index=False)
)

STRONG DECOUPLING — INITIAL SCREENING

P75 Generation increase: 2.700 TWh
P75 Emissions reduction: 1.453 MtCO2e

Số observations đạt cả 2 điều kiện: 106
Tỷ lệ trên 669 decoupling observations: 15.84%

--- STRONG DECOUPLING ---
                Area  Year                Ember region  Generation_increase_abs  Generation YoY change (%)  Emissions_reduction_abs  Emissions YoY change (%)  Emissions intensity (gCO2e/kWh)
               China  2015                        Asia                   17.551                      0.304                  138.837                    -3.452                          669.705
       United States  2016               North America                    3.570                      0.087                   86.289                    -4.184                          483.393
               China  2025                        Asia                  480.732                      4.795                   53.681                    -0.958                          528.278
         

In [17]:
# ============================================================
# - Trong 106 trường hợp strong decoupling ban đầu, bao nhiêu trường hợp thực sự giảm carbon intensity?
# - Strong decoupling có đồng thời thể hiện: Generation ↑ + Emissions ↓ + Intensity ↓ hay không?
# - Kiểm tra xem giảm emissions có đi cùng cải thiện hiệu quả carbon của hệ thống điện hay không.
#
# Không đặt thêm threshold trước: Trước tiên đo lường mức thay đổi intensity của 106 cases,
# sau đó mới quyết định tiêu chí strong decoupling cuối cùng.
# ============================================================

strong_check = strong_decoupling.copy()

# ------------------------------------------------------------
# 1. Lấy Total generation để xác định intensity của năm trước
# ------------------------------------------------------------

total_rows = df[
    (df["Area type"] == "Country or economy")
    & (df["Electricity source"] == "Total generation")
].copy()

total_rows = total_rows.sort_values(
    ["Area", "Year"]
)

total_rows["Intensity_previous"] = (
    total_rows
    .groupby("Area")["Emissions intensity (gCO2e/kWh)"]
    .shift(1)
)

# Chỉ lấy intensity năm trước
intensity_previous = total_rows[
    [
        "Area",
        "Year",
        "Intensity_previous"
    ]
].copy()

# ------------------------------------------------------------
# 2. Merge intensity năm trước
#    KHÔNG merge intensity hiện tại vì strong_check đã có rồi.
# ------------------------------------------------------------

strong_check = strong_check.merge(
    intensity_previous,
    on=["Area", "Year"],
    how="left"
)

# ------------------------------------------------------------
# 3. Tính thay đổi carbon intensity
# ------------------------------------------------------------

strong_check["Delta_Intensity"] = (
    strong_check["Emissions intensity (gCO2e/kWh)"]
    - strong_check["Intensity_previous"]
)

strong_check["Intensity_decreased"] = (
    strong_check["Delta_Intensity"] < 0
)

# ------------------------------------------------------------
# 4. Kết quả
# ------------------------------------------------------------

print("=" * 110)
print("STRONG DECOUPLING — INTENSITY VALIDATION")
print("=" * 110)

print(
    "\nSố strong-decoupling observations:",
    len(strong_check)
)

print("\n--- Carbon intensity change ---")

print(
    strong_check["Delta_Intensity"]
    .describe(
        percentiles=[.25, .50, .75, .90, .95]
    )
    .to_string()
)

print("\n--- Intensity direction ---")

print(
    strong_check["Intensity_decreased"]
    .value_counts()
    .rename({
        True: "Intensity ↓",
        False: "Intensity ↑ / unchanged"
    })
    .to_string()
)

print(
    "\nTỷ lệ intensity giảm:",
    f"{strong_check['Intensity_decreased'].mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# 5. Top cases giảm intensity mạnh nhất
# ------------------------------------------------------------

print("\nTOP 20 CASES GIẢM CARBON INTENSITY MẠNH NHẤT:")

print(
    strong_check[
        [
            "Area",
            "Year",
            "Ember region",
            "Generation_increase_abs",
            "Emissions_reduction_abs",
            "Generation YoY change (%)",
            "Emissions YoY change (%)",
            "Intensity_previous",
            "Emissions intensity (gCO2e/kWh)",
            "Delta_Intensity"
        ]
    ]
    .sort_values("Delta_Intensity")
    .head(20)
    .to_string(index=False)
)

STRONG DECOUPLING — INTENSITY VALIDATION

Số strong-decoupling observations: 106

--- Carbon intensity change ---
count    106.000000
mean     -32.249481
std       23.387586
min     -135.612000
25%      -45.758000
50%      -26.066500
75%      -16.351750
90%       -8.997500
95%       -5.476000
max       -3.688000

--- Intensity direction ---
Intensity_decreased
Intensity ↓    106

Tỷ lệ intensity giảm: 100.00%

TOP 20 CASES GIẢM CARBON INTENSITY MẠNH NHẤT:
                Area  Year                Ember region  Generation_increase_abs  Emissions_reduction_abs  Generation YoY change (%)  Emissions YoY change (%)  Intensity_previous  Emissions intensity (gCO2e/kWh)  Delta_Intensity
            Portugal  2010                      Europe                     4.10                    5.396                      8.303                   -24.132             452.815                          317.203         -135.612
            Portugal  2013                      Europe                     4.91     

In [18]:
# ============================================================
# Strong decoupling xảy ra cùng với sự thay đổi nào trong cơ cấu nguồn điện?
#
#   So với năm trước, các strong-decoupling events có xu hướng:
#   - giảm Fossil?
#   - tăng Renewables?
#   - tăng Solar / Wind / Hydro?
#   - hay chủ yếu đến từ Nuclear?
# ============================================================

strong_mix = strong_check[
    ["Area", "Year"]
].copy()

# ------------------------------------------------------------
# Lấy energy mix của năm hiện tại và năm trước
# ------------------------------------------------------------

mix_sources = [
    "Fossil",
    "Renewables",
    "Solar",
    "Wind",
    "Hydro",
    "Nuclear"
]

mix_rows = df[
    (df["Area type"] == "Country or economy")
    & (df["Electricity source"].isin(mix_sources))
].copy()

mix_rows = mix_rows[
    [
        "Area",
        "Year",
        "Electricity source",
        "Share of generation (%)"
    ]
]

# ------------------------------------------------------------
# Chuyển source thành columns
# ------------------------------------------------------------

mix_wide = mix_rows.pivot_table(
    index=["Area", "Year"],
    columns="Electricity source",
    values="Share of generation (%)",
    aggfunc="first"
).reset_index()

# ------------------------------------------------------------
# Lấy mix của năm trước
# ------------------------------------------------------------

mix_wide = mix_wide.sort_values(
    ["Area", "Year"]
)

for source in mix_sources:
    mix_wide[f"{source}_previous"] = (
        mix_wide
        .groupby("Area")[source]
        .shift(1)
    )

# ------------------------------------------------------------
# Chỉ lấy strong-decoupling events
# ------------------------------------------------------------

strong_mix = strong_mix.merge(
    mix_wide,
    on=["Area", "Year"],
    how="left"
)

# ------------------------------------------------------------
# Tính thay đổi share theo percentage points
# ------------------------------------------------------------

for source in mix_sources:
    strong_mix[f"Delta_{source}_pp"] = (
        strong_mix[source]
        - strong_mix[f"{source}_previous"]
    )

# ------------------------------------------------------------
# Tổng hợp
# ------------------------------------------------------------

delta_columns = [
    f"Delta_{source}_pp"
    for source in mix_sources
]

print("=" * 110)
print("ENERGY MIX TRANSITION DURING STRONG DECOUPLING")
print("=" * 110)

print(
    "\nSố strong-decoupling observations:",
    len(strong_mix)
)

print("\n--- Mean change (percentage points) ---")

print(
    strong_mix[delta_columns]
    .mean()
    .sort_values()
    .to_string()
)

print("\n--- Median change (percentage points) ---")

print(
    strong_mix[delta_columns]
    .median()
    .sort_values()
    .to_string()
)

# ------------------------------------------------------------
# Direction count
# ------------------------------------------------------------

print("\n--- Direction of energy-mix change ---")

for source in mix_sources:

    col = f"Delta_{source}_pp"

    increase = (strong_mix[col] > 0).sum()
    decrease = (strong_mix[col] < 0).sum()
    unchanged = (strong_mix[col] == 0).sum()

    print(
        f"{source:12s} | "
        f"↑ {increase:3d} | "
        f"↓ {decrease:3d} | "
        f"= {unchanged:3d}"
    )

ENERGY MIX TRANSITION DURING STRONG DECOUPLING

Số strong-decoupling observations: 106

--- Mean change (percentage points) ---
Delta_Fossil_pp       -4.083066
Delta_Wind_pp          0.393098
Delta_Solar_pp         0.547437
Delta_Nuclear_pp       1.116848
Delta_Hydro_pp         2.106904
Delta_Renewables_pp    2.976755

--- Median change (percentage points) ---
Delta_Fossil_pp       -3.3390
Delta_Solar_pp         0.0260
Delta_Wind_pp          0.0720
Delta_Nuclear_pp       0.1220
Delta_Hydro_pp         0.9495
Delta_Renewables_pp    1.9040

--- Direction of energy-mix change ---
Fossil       | ↑   4 | ↓ 102 | =   0
Renewables   | ↑  94 | ↓  12 | =   0
Solar        | ↑  63 | ↓   9 | =  31
Wind         | ↑  73 | ↓  11 | =  18
Hydro        | ↑  78 | ↓  26 | =   0
Nuclear      | ↑  54 | ↓  25 | =  26


Như vậy, sự tách biệt mạnh mẽ giữa sản lượng điện và lượng khí thải chủ yếu gắn liền với việc giảm tỷ lệ sản lượng điện từ nhiên liệu hóa thạch và tăng tỷ lệ sản lượng điện từ năng lượng tái tạo.

In [19]:
# ============================================================
# XÁC ĐỊNH HAI NHÓM ĐỂ SO SÁNH
#
# RQ4: Những trường hợp tăng sản lượng điện nhưng giảm phát thải (Strong Decoupling) 
# khác gì với những trường hợp tăng cả sản lượng điện và phát thải (Bad Decoupling)?
#
# Hai nhóm có quy mô mẫu bao nhiêu và có đủ dữ liệu để tiếp tục so sánh Energy Mix hay không?
#
# - Xác định:
#       Strong Decoupling = Generation ↑ & Emissions ↓
#       Bad Decoupling    = Generation ↑ & Emissions ↑
# - Chỉ kiểm tra quy mô mẫu ở bước này.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Chỉ lấy Country/Economy + Total generation và các biến carbon cần thiết
# ------------------------------------------------------------
dec = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"] == "Total generation")
].copy()

dec = dec[
    [
        "Area",
        "Year",
        "Generation (TWh)",
        "Emissions (MtCO2e)",
        "Emissions intensity (gCO2e/kWh)"
    ]
].sort_values(["Area", "Year"])

# ------------------------------------------------------------
# 2. Tính thay đổi giữa năm hiện tại và năm trước
# ------------------------------------------------------------
dec["Delta_Generation_TWh"] = (
    dec.groupby("Area")["Generation (TWh)"].diff()
)

dec["Delta_Emissions_MtCO2e"] = (
    dec.groupby("Area")["Emissions (MtCO2e)"].diff()
)

dec["Delta_Intensity"] = (
    dec.groupby("Area")["Emissions intensity (gCO2e/kWh)"].diff()
)

# Chỉ giữ country-year có đủ dữ liệu cho hai biến chính
dec_valid = dec.dropna(
    subset=["Delta_Generation_TWh", "Delta_Emissions_MtCO2e"]
).copy()

# ------------------------------------------------------------
# 3. Xác định hai nhóm
#
# Strong Decoupling:
#   Generation tăng + Emissions giảm
#
# Bad Decoupling:
#   Generation tăng + Emissions tăng
# ------------------------------------------------------------
strong_decoupling = dec_valid[
    (dec_valid["Delta_Generation_TWh"] > 0) &
    (dec_valid["Delta_Emissions_MtCO2e"] < 0)
].copy()

bad_decoupling = dec_valid[
    (dec_valid["Delta_Generation_TWh"] > 0) &
    (dec_valid["Delta_Emissions_MtCO2e"] > 0)
].copy()

# ------------------------------------------------------------
# 4. In kết quả kiểm tra
# ------------------------------------------------------------
print("\n--- Số quan sát ---")
print(f"Country-year observations hợp lệ : {len(dec_valid):,}")
print(f"Strong Decoupling                : {len(strong_decoupling):,}")
print(f"Bad Decoupling                   : {len(bad_decoupling):,}")

print("\n--- Tỷ lệ trên các trường hợp Generation tăng ---")

generation_up = dec_valid[
    dec_valid["Delta_Generation_TWh"] > 0
]

print(f"Generation tăng: {len(generation_up):,}")

if len(generation_up) > 0:
    print(
        f"Strong Decoupling: "
        f"{len(strong_decoupling) / len(generation_up) * 100:.2f}%"
    )

    print(
        f"Bad Decoupling: "
        f"{len(bad_decoupling) / len(generation_up) * 100:.2f}%"
    )

# ------------------------------------------------------------
# 5. Kiểm tra nhanh intensity trong hai nhóm
# ------------------------------------------------------------
print("\n--- Carbon intensity ---")

print(
    "Strong Decoupling có intensity giảm:",
    (
        strong_decoupling["Delta_Intensity"] < 0
    ).mean() * 100,
    "%"
)

print(
    "Bad Decoupling có intensity giảm:",
    (
        bad_decoupling["Delta_Intensity"] < 0
    ).mean() * 100,
    "%"
)



--- Số quan sát ---
Country-year observations hợp lệ : 5,706
Strong Decoupling                : 669
Bad Decoupling                   : 3,096

--- Tỷ lệ trên các trường hợp Generation tăng ---
Generation tăng: 3,816
Strong Decoupling: 17.53%
Bad Decoupling: 81.13%

--- Carbon intensity ---
Strong Decoupling có intensity giảm: 100.0 %
Bad Decoupling có intensity giảm: 41.56976744186046 %


In [21]:
# ============================================================
# SO SÁNH ENERGY MIX GIỮA STRONG vs BAD
#
# Energy Mix của hai nhóm thay đổi theo hướng nào?
#
# Purpose:
# - Load 2 nhóm đã xác định ở bước trước.
# - Ghép với thay đổi Energy Mix theo country-year.
# - So sánh mean, median và hướng thay đổi.
# - Chưa chạy statistical test ở bước này.
# ============================================================

from pathlib import Path
import pickle
import pandas as pd

dec = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"] == "Total generation")
].copy()

dec = dec[
    [
        "Area",
        "Year",
        "Generation (TWh)",
        "Emissions (MtCO2e)"
    ]
].sort_values(["Area", "Year"])

dec["Delta_Generation_TWh"] = (
    dec.groupby("Area")["Generation (TWh)"].diff()
)

dec["Delta_Emissions_MtCO2e"] = (
    dec.groupby("Area")["Emissions (MtCO2e)"].diff()
)

dec = dec.dropna(
    subset=["Delta_Generation_TWh", "Delta_Emissions_MtCO2e"]
)

generation_up = dec[
    dec["Delta_Generation_TWh"] > 0
].copy()

strong = generation_up[
    generation_up["Delta_Emissions_MtCO2e"] < 0
].copy()

bad = generation_up[
    generation_up["Delta_Emissions_MtCO2e"] > 0
].copy()

# ------------------------------------------------------------
# 1. Lấy Energy Mix theo country-year
#
# Chỉ dùng component sources để tránh double counting.
# Không dùng Renewables/Fossil aggregate ở bước tính tổng, nhưng vẫn có thể lấy chúng như indicator đã có sẵn.
# ------------------------------------------------------------
mix_sources = [
    "Fossil",
    "Renewables",
    "Solar",
    "Wind",
    "Hydro",
    "Nuclear"
]

mix = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"].isin(mix_sources))
][
    [
        "Area",
        "Year",
        "Electricity source",
        "Share of generation (%)"
    ]
].copy()

mix_pivot = (
    mix
    .pivot_table(
        index=["Area", "Year"],
        columns="Electricity source",
        values="Share of generation (%)",
        aggfunc="first"
    )
    .reset_index()
)

# ------------------------------------------------------------
# 2. Tính thay đổi percentage points so với năm trước
# ------------------------------------------------------------
for source in mix_sources:
    mix_pivot[f"Delta_{source}_pp"] = (
        mix_pivot
        .groupby("Area")[source]
        .diff()
    )

# ------------------------------------------------------------
# 3. Ghép với Strong / Bad
# ------------------------------------------------------------
key = ["Area", "Year"]

strong_mix = strong[key].merge(
    mix_pivot[
        key + [f"Delta_{s}_pp" for s in mix_sources]
    ],
    on=key,
    how="left"
)

bad_mix = bad[key].merge(
    mix_pivot[
        key + [f"Delta_{s}_pp" for s in mix_sources]
    ],
    on=key,
    how="left"
)

# ------------------------------------------------------------
# 4. Hàm thống kê mô tả
# ------------------------------------------------------------
def summarize_mix(data, group_name):
    rows = []

    for source in mix_sources:
        col = f"Delta_{source}_pp"

        rows.append({
            "Group": group_name,
            "Source": source,
            "N": data[col].notna().sum(),
            "Mean_pp": data[col].mean(),
            "Median_pp": data[col].median(),
            "Increase_%": (data[col] > 0).mean() * 100,
            "Decrease_%": (data[col] < 0).mean() * 100,
            "No_change_%": (data[col] == 0).mean() * 100
        })

    return pd.DataFrame(rows)

summary_mix = pd.concat(
    [
        summarize_mix(strong_mix, "Strong Decoupling"),
        summarize_mix(bad_mix, "Bad Decoupling")
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 5. In kết quả
# ------------------------------------------------------------
print("=== ENERGY MIX COMPARISON ===")

print("\n--- SAMPLE SIZE ---")
print(f"Strong Decoupling: {len(strong_mix):,}")
print(f"Bad Decoupling   : {len(bad_mix):,}")

print("\n--- MEAN CHANGE (percentage points) ---")
print(
    summary_mix[
        ["Group", "Source", "Mean_pp"]
    ].pivot(
        index="Source",
        columns="Group",
        values="Mean_pp"
    ).round(3)
)

print("\n--- MEDIAN CHANGE (percentage points) ---")
print(
    summary_mix[
        ["Group", "Source", "Median_pp"]
    ].pivot(
        index="Source",
        columns="Group",
        values="Median_pp"
    ).round(3)
)

print("\n--- DIRECTION OF CHANGE (%) ---")

for group in ["Strong Decoupling", "Bad Decoupling"]:
    print(f"\n{group}")

    temp = summary_mix[
        summary_mix["Group"] == group
    ].copy()

    for _, row in temp.iterrows():
        print(
            f"{row['Source']:12s} | "
            f"↑ {row['Increase_%']:6.2f}% | "
            f"↓ {row['Decrease_%']:6.2f}% | "
            f"= {row['No_change_%']:6.2f}%"
        )


=== ENERGY MIX COMPARISON ===

--- SAMPLE SIZE ---
Strong Decoupling: 669
Bad Decoupling   : 3,096

--- MEAN CHANGE (percentage points) ---
Group       Bad Decoupling  Strong Decoupling
Source                                       
Fossil               0.895             -5.027
Hydro               -1.037              3.888
Nuclear             -0.095              0.345
Renewables          -0.822              4.709
Solar                0.143              0.390
Wind                 0.057              0.426

--- MEDIAN CHANGE (percentage points) ---
Group       Bad Decoupling  Strong Decoupling
Source                                       
Fossil               0.000             -3.200
Hydro               -0.003              1.990
Nuclear              0.000              0.000
Renewables           0.000              2.836
Solar                0.000              0.000
Wind                 0.000              0.000

--- DIRECTION OF CHANGE (%) ---

Strong Decoupling
Fossil       | ↑   3.44% | ↓ 

In [ ]:
# - So sánh 6 nguồn điện giữa 669 Strong và 3,096 Bad observations.
# - Mann–Whitney U: kiểm định khác biệt phân phối.
# - Cohen's d: đo độ lớn khác biệt.
# - Xác định nguồn nào thực sự đặc trưng cho Strong Decoupling.
# - Ưu tiên Renewable/Fossil/Hydro; vẫn kiểm tra Solar/Wind/Nuclear để không bỏ sót biến.

from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd

mix_cols = [
    "Delta_Fossil_pp",
    "Delta_Renewables_pp",
    "Delta_Hydro_pp",
    "Delta_Solar_pp",
    "Delta_Wind_pp",
    "Delta_Nuclear_pp"
]

# Đúng nguồn dữ liệu:
# strong_mix = 669 Strong Decoupling observations
# bad_mix    = 3,096 Bad Decoupling observations
strong_test = strong_mix[mix_cols].dropna()
bad_test = bad_mix[mix_cols].dropna()

results = []

for col in mix_cols:
    x = strong_test[col].values
    y = bad_test[col].values

    # Mann–Whitney U
    u_stat, p_value = mannwhitneyu(
        x,
        y,
        alternative="two-sided"
    )

    # Cohen's d
    nx = len(x)
    ny = len(y)

    pooled_sd = np.sqrt(
        (
            (nx - 1) * np.var(x, ddof=1)
            + (ny - 1) * np.var(y, ddof=1)
        )
        / (nx + ny - 2)
    )

    d = (np.mean(x) - np.mean(y)) / pooled_sd

    results.append({
        "Source": col.replace("Delta_", "").replace("_pp", ""),
        "Strong_mean": np.mean(x),
        "Bad_mean": np.mean(y),
        "Mean_difference": np.mean(x) - np.mean(y),
        "MW_U": u_stat,
        "MW_p": p_value,
        "Cohens_d": d,
        "abs_Cohens_d": abs(d)
    })

stat_results = pd.DataFrame(results)

# Xếp hạng theo effect size
stat_results = stat_results.sort_values(
    "abs_Cohens_d",
    ascending=False
).reset_index(drop=True)

print("=== STATISTICAL COMPARISON: STRONG vs BAD ===")

print(f"\nStrong Decoupling: {len(strong_test):,}")
print(f"Bad Decoupling   : {len(bad_test):,}")

print("\n--- RESULTS ---")
print(
    stat_results[
        [
            "Source",
            "Strong_mean",
            "Bad_mean",
            "Mean_difference",
            "MW_p",
            "Cohens_d",
            "abs_Cohens_d"
        ]
    ].round(4).to_string(index=False)
)

print("\n--- EFFECT SIZE GUIDE ---")
print("|d| < 0.2   : negligible")
print("|d| < 0.5   : small")
print("|d| < 0.8   : medium")
print("|d| >= 0.8  : large")

print("\n--- SIGNIFICANCE ---")
print("p < 0.05 → statistically significant")

=== STATISTICAL COMPARISON: STRONG vs BAD ===

Strong Decoupling: 593
Bad Decoupling   : 2,667

--- RESULTS ---
    Source  Strong_mean  Bad_mean  Mean_difference  MW_p  Cohens_d  abs_Cohens_d
    Fossil      -4.7522    0.8984          -5.6505   0.0   -1.3798        1.3798
Renewables       4.4731   -0.7992           5.2723   0.0    1.2885        1.2885
     Hydro       3.6252   -0.9952           4.6204   0.0    1.1487        1.1487
      Wind       0.4334    0.0591           0.3743   0.0    0.4075        0.4075
   Nuclear       0.2791   -0.0991           0.3782   0.0    0.3612        0.3612
     Solar       0.3917    0.1463           0.2454   0.0    0.2403        0.2403

--- EFFECT SIZE GUIDE ---
|d| < 0.2   : negligible
|d| < 0.5   : small
|d| < 0.8   : medium
|d| >= 0.8  : large

--- SIGNIFICANCE ---
p < 0.05 → statistically significant


In [27]:
# Quality check: Các observation bị loại khỏi kiểm định Strong vs Bad
# có tập trung ở một số biến cơ cấu nguồn điện hay không?         
#
# - Kiểm tra nguyên nhân 669 → 593 và 3,096 → 2,667.
# - Đảm bảo kết quả statistical comparison không bị ảnh hưởng bất thường bởi missing data.

print("=== MISSING DATA CHECK: STRONG vs BAD ===")

for col in mix_cols:
    strong_missing = strong_mix[col].isna().sum()
    bad_missing = bad_mix[col].isna().sum()

    print(
        f"{col:<25} | "
        f"Strong missing: {strong_missing:>4} "
        f"({strong_missing / len(strong_mix) * 100:6.2f}%) | "
        f"Bad missing: {bad_missing:>4} "
        f"({bad_missing / len(bad_mix) * 100:6.2f}%)"
    )

print("\n--- COMPLETE CASES ---")
print(f"Strong ban đầu : {len(strong_mix):,}")
print(f"Strong dùng test: {len(strong_test):,}")
print(f"Bad ban đầu    : {len(bad_mix):,}")
print(f"Bad dùng test  : {len(bad_test):,}")

=== MISSING DATA CHECK: STRONG vs BAD ===
Delta_Fossil_pp           | Strong missing:    0 (  0.00%) | Bad missing:    0 (  0.00%)
Delta_Renewables_pp       | Strong missing:    0 (  0.00%) | Bad missing:    7 (  0.23%)
Delta_Hydro_pp            | Strong missing:    9 (  1.35%) | Bad missing:   93 (  3.00%)
Delta_Solar_pp            | Strong missing:    8 (  1.20%) | Bad missing:   53 (  1.71%)
Delta_Wind_pp             | Strong missing:   41 (  6.13%) | Bad missing:  161 (  5.20%)
Delta_Nuclear_pp          | Strong missing:   51 (  7.62%) | Bad missing:  318 ( 10.27%)

--- COMPLETE CASES ---
Strong ban đầu : 669
Strong dùng test: 593
Bad ban đầu    : 3,096
Bad dùng test  : 2,667


In [29]:
# Robustness: Kết luận Strong vs Bad có phụ thuộc vào việc hai nhóm có kích thước rất khác nhau hay không?"
#
# Purpose:
# - Cân bằng kích thước mẫu: 593 Strong vs 593 Bad.
# - Lặp nhiều lần lấy mẫu ngẫu nhiên từ Bad.
# - Kiểm tra effect size trung bình và độ ổn định của thứ hạng.

from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n_strong = len(strong_test)
n_iter = 500

robust_results = []

for i in range(n_iter):

    # Lấy ngẫu nhiên cùng số lượng observation từ Bad
    bad_sample = bad_test.sample(
        n=n_strong,
        random_state=int(rng.integers(0, 1_000_000))
    )

    for col in mix_cols:

        x = strong_test[col].values
        y = bad_sample[col].values

        # Cohen's d
        nx = len(x)
        ny = len(y)

        pooled_sd = np.sqrt(
            (
                (nx - 1) * np.var(x, ddof=1)
                + (ny - 1) * np.var(y, ddof=1)
            )
            / (nx + ny - 2)
        )

        d = (np.mean(x) - np.mean(y)) / pooled_sd

        robust_results.append({
            "Iteration": i + 1,
            "Source": col.replace("Delta_", "").replace("_pp", ""),
            "Cohens_d": d
        })

robust_results = pd.DataFrame(robust_results)

summary_robust = (
    robust_results
    .groupby("Source")["Cohens_d"]
    .agg(
        Mean_d="mean",
        Median_d="median",
        Std_d="std",
        Min_d="min",
        Max_d="max"
    )
    .sort_values("Mean_d", key=lambda x: x.abs(), ascending=False)
)

print("=== ROBUSTNESS CHECK: BALANCED SAMPLING ===")
print(f"Strong sample size : {n_strong:,}")
print(f"Bad sample size    : {n_strong:,}")
print(f"Iterations         : {n_iter:,}")

print("\n--- COHEN'S d ACROSS 500 RANDOM SAMPLES ---")
print(summary_robust.round(4).to_string())

print("\n--- INTERPRETATION GUIDE ---")
print("Nếu thứ hạng effect size ổn định qua 500 lần: Kết luận Strong vs Bad không phụ thuộc chủ yếu vào sample-size imbalance.")

=== ROBUSTNESS CHECK: BALANCED SAMPLING ===
Strong sample size : 593
Bad sample size    : 593
Iterations         : 500

--- COHEN'S d ACROSS 500 RANDOM SAMPLES ---
            Mean_d  Median_d   Std_d   Min_d   Max_d
Source                                              
Fossil     -1.1958   -1.1974  0.0320 -1.2823 -1.0962
Renewables  1.1099    1.1118  0.0306  1.0218  1.1924
Hydro       0.9662    0.9672  0.0249  0.9043  1.0285
Wind        0.3390    0.3376  0.0308  0.2512  0.4114
Nuclear     0.2863    0.2855  0.0211  0.2312  0.3421
Solar       0.2292    0.2275  0.0481  0.1078  0.3457

--- INTERPRETATION GUIDE ---
Nếu thứ hạng effect size ổn định qua 500 lần: Kết luận Strong vs Bad không phụ thuộc chủ yếu vào sample-size imbalance.


Các trường hợp tăng sản lượng điện nhưng đồng thời giảm phát thải carbon có sự chuyển dịch cơ cấu nguồn điện rõ rệt khỏi nhiên liệu hóa thạch và hướng về năng lượng tái tạo, trong đó Hydro thể hiện khác biệt lớn nhất sau Fossil và Renewables. Ngược lại, Solar và Wind có effect size nhỏ hơn trong việc phân biệt hai nhóm.

In [30]:
# NOTE 08 — Final checkpoint

from pathlib import Path
import pickle

checkpoint_dir = project_root / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

note08_checkpoint = {
    # RQ4 groups
    "strong": strong,
    "bad_mix": bad_mix,

    # Energy-mix comparison
    "strong_mix": strong_mix,
    "summary_mix": summary_mix,

    # Statistical comparison
    "stat_results": stat_results,

    # Robustness analysis
    "robust_results": robust_results,
    "summary_robust": summary_robust,

    # Earlier decoupling results
    "decoupling": decoupling,
    "decoupling_up_down": decoupling_up_down,
    "strong_decoupling": strong_decoupling,
    "strong_check": strong_check,
}

checkpoint_path = checkpoint_dir / "note08_carbon_transition_rq4.pkl"

with open(checkpoint_path, "wb") as f:
    pickle.dump(note08_checkpoint, f)

print("=== NOTE 08 CHECKPOINT ===")
print("Đã lưu toàn bộ kết quả RQ4 sau khi hoàn tất kiểm định.")
print(f"File: {checkpoint_path}")
print(f"Số objects: {len(note08_checkpoint)}")
print("\nCác nhóm kết quả chính:")
print("- Decoupling")
print("- Strong vs Bad")
print("- Energy-mix comparison")
print("- Mann–Whitney + Cohen's d")
print("- 500-iteration robustness check")

=== NOTE 08 CHECKPOINT ===
Đã lưu toàn bộ kết quả RQ4 sau khi hoàn tất kiểm định.
File: d:\phan tich data\Ember_Electricity\checkpoints\note08_carbon_transition_rq4.pkl
Số objects: 11

Các nhóm kết quả chính:
- Decoupling
- Strong vs Bad
- Energy-mix comparison
- Mann–Whitney + Cohen's d
- 500-iteration robustness check
